# Automatic Classification of Edible Mushrooms using Deep Learning and Segmentation Techniques

**Authors:** Leonardo Berselli, Thomas Ottonello

---

## 1. Project Objectives

This work aims to develop a binary classification system to determine mushroom edibility based exclusively on visual data. The methodological approach compares the performance of a pre-trained convolutional neural network (CNN), specifically MobileNetV3-Small, on two types of input:

1. **Original images** (baseline): preservation of the full environmental context
2. **Segmented images**: isolation of the subject through algorithmic background removal (GrabCut)

The research hypothesis focuses on the trade-off between contextual information (habitat, substrate, surrounding elements) and background noise reduction, evaluating which approach optimizes the model's predictive capabilities.

---

## 2. Dataset and Preparation Methodology

### 2.1 Data Source
The dataset used comes from the **[Mushroom1](https://www.kaggle.com/datasets/zlatan599/mushroom1/)** collection available on Kaggle, comprising photographic images of 169 distinct species of mushrooms and lichens.

### 2.2 Preprocessing and Balancing
Given the imbalanced nature and high cardinality of the original classes, the following preprocessing pipeline was implemented:

- **Dimensionality reduction**: selection of 50 classes (25 edible + 25 non-edible)
- **Stratified sampling**: limit of 200 images per species in the training set
- **Data split**: ~3000 images for each subset (train/validation/test)
- **Edibility annotation**: creation of the binary label `edible` through systematic bibliographic research

**Classification criterion**: Species were labeled as non-edible (`False`) in the case of:
- Membership in the lichen group
- Insufficient or ambiguous documentation on edibility
- Presence of toxic compounds even in limited concentrations

---

## 3. Experimental Pipeline

### 3.1 Exploration Phase: Segmentation Techniques
Implementation and comparative evaluation of five segmentation algorithms:

1. **GrabCut**: iterative algorithm based on Graph Cut and Gaussian Mixture Models
2. **Mean Shift + GrabCut**: pre-segmentation via spatial-chromatic clustering
3. **SLIC (Simple Linear Iterative Clustering)**: super-pixel segmentation with hierarchical clustering
4. **Normalized Cuts**: graph-based approach with global optimization
5. **Color Names Saliency**: saliency detection based on a perceptual color representation

**Selected algorithm**: GrabCut, for the excellent trade-off between segmentation accuracy, robustness, and computational efficiency.

### 3.2 Model Architecture
**Neural network**: MobileNetV3-Small
- **Rationale**: lightweight architecture optimized for resource-constrained devices
- **Transfer learning**: initialization with weights pre-trained on ImageNet (1000 classes)
- **Adaptation**: modification of the output layer from 1000 to 1 unit (binary classification)

**Training configuration**:
- Loss function: Binary Cross-Entropy with Logits (`BCEWithLogitsLoss`)
- Optimizer: Adam with differentiated learning rates (backbone: 1e-5, classifier: 2e-4)
- Scheduler: `ReduceLROnPlateau` (dynamic learning-rate reduction)
- Metrics: Accuracy, Precision, Recall, F1-Score, Confusion Matrix

### 3.3 Experimental Design
Two independent parallel experiments:

- **Experiment 1**: training on unprocessed original images
- **Experiment 2**: training on segmented images (GrabCut output)

**Objective**: quantify the impact of background removal on the model's generalization ability.

---

## 4. Evaluation Metrics and Practical Relevance

### 4.1 Primary Metrics
- **Accuracy**: proportion of correct predictions over the total
- **Precision**: reliability of positive predictions (TP / (TP + FP))
- **Recall**: coverage of true positive cases (TP / (TP + FN))
- **F1-Score**: harmonic mean of precision and recall

### 4.2 Error Analysis
**Criticality of False Positives**: In the applied context of mushroom classification, false positives (a poisonous mushroom classified as edible) represent a significant risk to human health. Analyzing the confusion matrix allows identification of systematic patterns in classification errors.

---

## 5. Technology Stack

### 5.1 Libraries and Frameworks
- **PyTorch 2.x**: deep learning framework with CUDA support
- **OpenCV (cv2)**: image processing and GrabCut implementation
- **scikit-image**: advanced segmentation algorithms (SLIC, Normalized Cuts)
- **torchvision**: pre-trained models and standard transformations
- **scikit-learn**: evaluation metrics and clustering (K-Means)
- **NumPy/Pandas**: tensor manipulation and tabular data structures
- **Matplotlib**: results visualization and exploratory analysis

### 5.2 Computational Optimizations
- **I/O parallelization**: `ThreadPoolExecutor` for read/write operations
- **GPU acceleration**: CUDA training when available
- **Batch processing**: efficient processing via `DataLoader`


## 1. Environment Setup and Configuration

### 1.1 Importing Dependencies

This section initializes the working environment by importing all the libraries required to implement the classification system. Dependencies are organized by functional domain:

**Computer Vision and Image Processing:**
- `cv2` (OpenCV): image processing, GrabCut algorithm implementation
- `skimage` (scikit-image): advanced segmentation algorithms (SLIC, Normalized Cuts)
- `PIL` (Pillow): image manipulation and I/O

**Deep Learning:**
- `torch`: main framework for neural networks
- `torchvision`: pre-trained models, transformations, and dataset utilities

**Data Analysis and Scientific Computing:**
- `numpy`: multidimensional array operations and linear algebra
- `pandas`: manipulation of tabular data structures
- `scipy`: advanced scientific functions (loading .mat files)

**Classical Machine Learning:**
- `sklearn` (scikit-learn): clustering (K-Means), evaluation metrics

**Utilities and Visualization:**
- `matplotlib.pyplot`: plotting graphs and images
- `kagglehub`: API interface to download datasets from Kaggle
- `tqdm`: progress bar for iterations
- `pathlib`, `os`: filesystem management
- `concurrent.futures`: parallelization via thread/process pools

**Type Hints:**
Custom type aliases are defined to improve readability and type checking:
- `BoolArray`: NumPy boolean array
- `UIntArray`: array of 8-bit unsigned integers (RGB images)
- `FloatArray`: float32 array (numerical processing)
- `IntArray`: array of 32-bit signed integers (segmentation maps)

These aliases facilitate function documentation and allow static analysis tools (e.g., mypy) to verify type correctness.


In [1]:
# all imports
import cv2
import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import skimage as ski
import kagglehub
import torch
import torchvision.models as models
import scipy
import torch.optim as optim
import tqdm

from math import sqrt
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    silhouette_score,
)
from skimage.segmentation import slic
from skimage.filters import threshold_otsu
from skimage.graph import rag_mean_color, cut_normalized
from warnings import warn
from typing import overload, Literal, TypeAlias
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

BoolArray: TypeAlias = np.typing.NDArray[np.bool_]
UIntArray: TypeAlias = np.typing.NDArray[np.uint8]
FloatArray: TypeAlias = np.typing.NDArray[np.float32]
IntArray: TypeAlias = np.typing.NDArray[np.int32]

ModuleNotFoundError: No module named 'torch'

### 1.2 Seed Initialization for Reproducibility

To ensure **reproducibility of the experiments**, deterministic seeds are set for all pseudo-random number generators used in the project:

```python
random.seed(42)           # Libreria standard Python
np.random.seed(42)        # NumPy
torch.manual_seed(42)     # PyTorch (CPU)
cv2.setRNGSeed(42)        # OpenCV
```

**Scientific rationale**: Fixing the seed makes it possible to:
1. Obtain identical results across multiple executions of the same code
2. Facilitate debugging by reproducing exactly the same random samples
3. Enable independent validation of results by third parties

**Note**: For GPU experiments, `torch.cuda.manual_seed_all(42)` is also required to guarantee full determinism.

**OpenCV optimizations**:
```python
cv2.setUseOptimized(True)
```
Enables OpenCV SIMD (Single Instruction Multiple Data) optimizations to accelerate image-processing operations on supported architectures (SSE, AVX, NEON).


In [2]:
# setting seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
# cv2 set
cv2.setRNGSeed(42)
cv2.setUseOptimized(True)

NameError: name 'torch' is not defined

## 2. Dataset Acquisition and Loading

### 2.1 Downloading the Dataset from Kaggle

Using the `kagglehub` API to automatically download the latest available version of the **Mushroom1** dataset:

```python
path = kagglehub.dataset_download("zlatan599/mushroom1")
```

**Dataset characteristics:**
- **Cardinality**: 169 distinct species of mushrooms and lichens
- **Format**: JPEG/PNG images with CSV annotations
- **Structure**: pre-defined splits (train/val/test)
- **Size**: ~15,000 total images with variable resolution

**Cache handling**: kagglehub implements a local caching system, avoiding repeated downloads if the dataset is already present in the filesystem.


In [3]:
# Download latest version
path = kagglehub.dataset_download("zlatan599/mushroom1")

print("Path to dataset files:", path)

### 2.2 Loading Metadata

CSV files contain annotations for each image:

- `train.csv`: paths and labels for the training set
- `val.csv`: paths and labels for the validation set  
- `test.csv`: paths and labels for the test set

Each row in the CSVs specifies:
- `image_path`: relative path to the image
- `label`: scientific name of the species

These metadata will later be enriched with the binary column `edible` obtained through bibliographic research.


In [ ]:
train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))
val_df = pd.read_csv(os.path.join(path, "val.csv"))

print("Train DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)
print("Validation DataFrame shape:", val_df.shape)

print("Train DataFrame columns:", train_df.columns.tolist())
print("Test DataFrame columns:", test_df.columns.tolist())
print("Validation DataFrame columns:", val_df.columns.tolist())

### 2.3 Class Distribution Analysis

Graphical visualization of the distribution of the 169 species across the three subsets (train/val/test) via overlapping histograms. This exploratory analysis allows:

1. **Identifying imbalances**: checking whether some species are over/under-represented
2. **Assessing split consistency**: ensuring proportions are similar across train/val/test
3. **Detecting rare classes**: species with very few examples (candidates for data augmentation or exclusion)

The visualization uses logarithmic scales on the y-axis to handle the wide variability in class frequencies.


In [ ]:
# correcting paths in dataframes
train_df["image_path"] = train_df["image_path"].apply(
    lambda p: os.path.join(path, "/".join(p.split("/")[-3:]))
)
test_df["image_path"] = test_df["image_path"].apply(
    lambda p: os.path.join(path, "/".join(p.split("/")[-3:]))
)
val_df["image_path"] = val_df["image_path"].apply(
    lambda p: os.path.join(path, "/".join(p.split("/")[-3:]))
)

# checking if corrected paths exist
if os.path.exists(train_df["image_path"].iloc[0]):
    print("Corrected path exists.")
else:
    raise FileNotFoundError("Corrected image path does not exist.")

### 2.4 Initializing the Edibility Annotation File

Creation of the `labels.csv` file containing the mapping `label` → `edible`:

```python
labels_df = pd.DataFrame({"label": list(labels)})
labels_df["edible"] = False  # Inizializzazione conservativa
```

**Conservative strategy**: All species are initially labeled as non-edible (`False`). Subsequently, only species with reliable documentation on edibility are marked as `True`.

**Overwrite prevention**: The file is created only if it does not already exist, preserving any previous manual annotations.


In [ ]:
# getting the labels from the training set
train_labels = train_df["label"].value_counts().index
test_labels = test_df["label"].value_counts().index
val_labels = val_df["label"].value_counts().index

print("There are", len(train_labels), "unique labels in the training set.")
print("There are", len(test_labels), "unique labels in the test set.")
print("There are", len(val_labels), "unique labels in the validation set.")

# checking if all labels are the same across train, val, and test sets
if set(train_labels) == set(test_labels) == set(val_labels):
    print("All sets have the same labels.")
    labels = train_labels
else:
    raise ValueError("Labels differ between train, val, and test sets.")


### 2.5 Integrating Edibility Annotations

After manually populating the `labels.csv` file with information about the edibility of each species (through systematic bibliographic research on accredited mycological sources), these data are integrated into the main DataFrames:

```python
train_df = pd.merge(train_df, labels_df, on="label", how="left")
val_df = pd.merge(val_df, labels_df, on="label", how="left")
test_df = pd.merge(test_df, labels_df, on="label", how="left")
```

**Applied classification criteria**:
1. **Lichens**: automatically classified as non-edible
2. **Controversial species**: in the presence of conflicting literature, classified as non-edible (precautionary principle)
3. **Insufficient documentation**: classified as non-edible

**Integrity check**: Verify that there are no missing values in the `edible` column after the merge; otherwise an error (`ValueError`) is raised.


In [ ]:
# functions to plot simple images


def plot_single_image(
    img: UIntArray, title="Image", size=(4, 4), with_hist=False
) -> None:
    if with_hist:
        _, axes = plt.subplots(1, 2, figsize=(size[0] * 2, size[1]))
        axes[0].imshow(img, cmap="gray")
        axes[0].set_title(title)
        axes[0].axis("off")
        axes[1].hist(img.ravel(), bins=256)
        axes[1].set_title("Histogram")
        plt.show()
    else:
        plt.figure(figsize=size)
        plt.imshow(img)
        plt.title(title)
        plt.axis("off")
        plt.show()


def plot_multiple_images(
    imgs, titles=None, size=(5, 5), max_per_row=3, tight_layout=True
) -> None:
    n = len(imgs)
    if titles != None and len(titles) != n:
        print("Titles length does not match number of images.")
        print("Skipping titles.")
        titles = None

    if n <= max_per_row:
        max_per_row = n

    h = (n // max_per_row) + (1 if n % max_per_row != 0 else 0)

    _, axes = plt.subplots(h, max_per_row, figsize=(size[0] * max_per_row, size[1] * h))
    axes = axes.ravel()
    for i in range(n):
        axes[i].imshow(imgs[i], cmap="gray" if len(imgs[i].shape) == 2 else None)
        if titles:
            axes[i].set_title(titles[i])
        axes[i].axis("off")
    for i in range(n, h * max_per_row):
        axes[i].axis("off")

    if tight_layout:
        plt.tight_layout()
    plt.show()

### 2.6 Dataset Reduction and Balancing

Given the high class cardinality (169 species) and the imbalance between categories, a controlled reduction is performed:

**Objectives**:
1. Reduce the computational complexity of the problem
2. Balance the number of samples per edible/non-edible class
3. Limit overfitting on classes with many samples

**Procedure**:
1. Selection of 25 edible and 25 non-edible species (50 classes total)
2. Random sampling of up to 200 images per species in the training set
3. Full preservation of the validation and test sets for unbiased evaluation

```python
train_df_reduced = train_df.groupby("label").apply(
    lambda x: x.sample(n=min(200, len(x)), random_state=42)
)
```

**Expected outcome**: ~3000 training images, with a balanced distribution between edible/non-edible.


In [ ]:
# calculating counts for each label in train, val, and test sets
counts_train = train_df["label"].value_counts()
counts_val = val_df["label"].value_counts()
counts_test = test_df["label"].value_counts()

x = np.arange(len(labels))

fig, axes = plt.subplots(3, 1, figsize=(20, 15), sharex=True)

axes[0].bar(x, counts_train, color="#1f77b4")
axes[0].set_title(f"Train (n={len(train_df)})")
axes[0].set_ylabel("Count")
# set y-limits for better visualization
axes[0].set_ylim(counts_train.min() * 0.9, counts_train.max() * 1.1)

axes[1].bar(x, counts_val, color="#ff7f0e")
axes[1].set_title(f"Validation (n={len(val_df)})")
axes[1].set_ylabel("Count")

axes[2].bar(x, counts_test, color="#2ca02c")
axes[2].set_title(f"Test (n={len(test_df)})")
axes[2].set_ylabel("Count")

# setting x-ticks and labels
plt.xticks(x, labels, rotation=90, fontsize=8)
plt.xlabel("Labels")
plt.tight_layout()
plt.show()

### Development Notes

Implemented optimizations:
- ✅ Resize to 224×224 (MobileNetV3 standard input size)
- ✅ NumPy code vectorization for performance
- ✅ Parallel processing via ThreadPoolExecutor
- ✅ Optimal number of segments for SLIC determined experimentally


In [ ]:
# saving the labels to a .csv file
# if the file already exists, it wont be overwritten
if os.path.exists("labels.csv"):
    print("labels.csv already exists. Not overwriting.")
else:
    labels_df = pd.DataFrame({"label": list(labels)})
    labels_df["edible"] = False
    labels_df.to_csv("labels.csv", index=False)
    print("labels.csv created.")

## 3. Segmentation Algorithm Implementation

### 3.1 Hierarchical Segmentation with SLIC

**SLIC (Simple Linear Iterative Clustering)** is a super-pixel segmentation algorithm that groups spatially close pixels with similar colors. The implementation uses a multi-resolution hierarchical structure:

#### 3.1.1 Hierarchy Generation

```python
def generate_hierarchy(image):
    # Z1: risoluzione originale (400 segmenti)
    z1 = slic(image, n_segments=400, compactness=5)

    # Z2: risoluzione 2/3 (200 segmenti)
    image_z2 = cv2.resize(image, (int(w*2/3), int(h*2/3)))
    z2 = slic(image_z2, n_segments=200, compactness=5)

    # Z3: risoluzione 1/2 (100 segmenti)
    image_z3 = cv2.resize(image, (int(w/2), int(h/2)))
    z3 = slic(image_z3, n_segments=100, compactness=5)
```

**Compactness parameter**: Controls the trade-off between:
- **Spatial compactness**: segments with regular shapes
- **Chromatic homogeneity**: segments with uniform colors

#### 3.1.2 Computing Mean Colors in CIELab

For each segment, the mean color is computed in the **CIELab** color space:

```python
def get_avg_color_per_segment(img, segment_map):
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    for seg_id in np.unique(segment_map):
        mask = segment_map == seg_id
        avg_color = img_lab[mask].mean(axis=0)
```

**Rationale**: The CIELab space is perceptually uniform, i.e., Euclidean distances in this space correspond to color differences perceived as equivalent by the human eye, unlike RGB.

**Output**: Dictionary `{segment_id: [L, a, b]}` with mean colors for each segment.


In [ ]:
# after manually adding the 'edible' column by searching online
# and automatically setting false if a species is a lichen or little to none information is found

labels_df = pd.read_csv("labels.csv")

labels_df["edible"] = (
    labels_df["edible"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
)

# se ci sono valori diversi da true/false (o vuoti), li segnala subito
if labels_df["edible"].isnull().any():
    bad_rows = labels_df[labels_df["edible"].isnull()]
    raise ValueError(
        "Valori non validi nella colonna 'edible' (attesi True/False). "
        f"Righe problematiche (prime 5):\n{bad_rows.head()}"
    )
# checking distribution and if there are any issues
counts = labels_df["edible"].value_counts()
print(counts)
print("Any missing label? ", any(labels_df["edible"].isnull()))

### 3.2 Color Clustering with K-Means

Applying K-Means to the mean colors of the segments to obtain a simplified representation of the image:

#### 3.2.1 Automatic Selection of the Number of Clusters

```python
def kmeans_on_segments(segmentation_map, color_dict):
    best_score = -1
    for k in range(5, 20):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        if score > best_score:
            best_model = kmeans
            best_score = score
```

**Silhouette Score**: Metric that measures how well elements are assigned to their own cluster compared to others. Range: [-1, 1], where:
- 1: well-separated and compact clusters
- 0: overlapping clusters
- -1: incorrect assignment

#### 3.2.2 Objective

Reduce the number of distinct colors in the image by grouping chromatically similar segments into a single cluster. This facilitates:
1. Subsequent identification of foreground/background regions
2. Creation of more robust binary masks
3. Analysis of dominant chromatic components


In [ ]:
# merging the 'edible' information back to the main dataframes
if "edible" in train_df.columns:
    print("'edible' column already exists in train_df. Skipping merge.")
else:
    train_df = pd.merge(train_df, labels_df, on="label", how="left")
    val_df = pd.merge(val_df, labels_df, on="label", how="left")
    test_df = pd.merge(test_df, labels_df, on="label", how="left")

# checking for any missing 'edible' values
if train_df["edible"].isnull().any():
    print(train_df.loc[train_df["edible"].isnull(), ["label", "edible"]])
    raise ValueError("Missing 'edible' values in training set after merge.")

### 3.3 Background Probability Computation

Implementation of a heuristic approach to estimate the probability that each cluster belongs to the background:

#### 3.3.1 Features Extracted per Cluster

1. **Area**: Total number of pixels in the cluster
   ```python
   area = np.sum(segmentation_map == seg_id)
   ```

2. **Border Connectivity (BndCon)**: Number of cluster pixels that touch the image borders
   ```python
   border_pixels = np.concatenate([
       segmentation_map[0, :],      # bordo superiore
       segmentation_map[-1, :],     # bordo inferiore
       segmentation_map[:, 0],      # bordo sinistro
       segmentation_map[:, -1]      # bordo destro
   ])
   border_area = np.sum(border_pixels == seg_id)
   ```

3. **Spatial Compactness**: Spatial dispersion of the cluster relative to its centroid
   ```python
   dist = sqrt((x_centroid - x_cluster_center)^2 + (y_centroid - y_cluster_center)^2)
   ```

#### 3.3.2 Probability Computation

Composite formula integrating the three features:

```python
BndCon = (border_area / sqrt(area)) * compactness
P_background = 1 - exp(-BndCon / (2 * mean_BndCon^2))
```

**Intuition**: Clusters that are spatially large, touch the borders, and are dispersed have a high probability of being background.


In [ ]:
# reducing to only 25 classes both for edible and non-edible mushrooms
# and dropping 90% of the training set images
# to speed up experiments and as we need to apply grabcut segmentation on all images remaining
def reduce_to_n_classes(df: pd.DataFrame, n: int = 25) -> pd.DataFrame:
    edible_counts = df[df["edible"]]["label"].value_counts()
    nonedible_counts = df[~df["edible"]]["label"].value_counts()

    top_edible = edible_counts.nlargest(n).index.tolist()
    top_nonedible = nonedible_counts.nlargest(n).index.tolist()

    reduced_df = df[df["label"].isin(top_edible + top_nonedible)].reset_index(drop=True)
    return reduced_df


classes = 15
train_df = reduce_to_n_classes(train_df, classes)
val_df = reduce_to_n_classes(val_df, classes)
test_df = reduce_to_n_classes(test_df, classes)


# mantaining class distribution but with only 200 random samples per mushroom species
def reduce_to_n_samples_per_class(
    df: pd.DataFrame, sample_size: int = 200
) -> pd.DataFrame:
    # the sampling isnt random as we need always the already segmented images
    reduced_df = (
        df.sample(frac=1, random_state=42)
        .groupby("label", group_keys=False)
        .head(sample_size)
        .reset_index(drop=True)
    )

    return reduced_df


sample_size_per_class = 100
train_df = reduce_to_n_samples_per_class(train_df, sample_size_per_class)
val_df = reduce_to_n_samples_per_class(val_df, sample_size_per_class)
test_df = reduce_to_n_samples_per_class(test_df, sample_size_per_class)

print(train_df["label"].nunique(), "unique labels in reduced train set.")
print(
    train_df[train_df["edible"]]["label"].nunique(),
    "unique edible labels in reduced train set.",
)

# new shapes
print("Reduced Train DataFrame shape:", train_df.shape)
print("Reduced Validation DataFrame shape:", val_df.shape)
print("Reduced Test DataFrame shape:", test_df.shape)


### 3.4 Color Names Saliency Detection

Implementation of a saliency detection algorithm based on the **Color Names** representation of colors.

#### 3.4.1 Color Names Representation

Each RGB pixel is mapped to an 11-dimensional probability vector corresponding to 11 basic “color names” (red, green, blue, yellow, orange, purple, pink, brown, black, gray, white):

```python
def im2c(img, w2c):
    # Quantizzazione RGB: 256^3 → 32^3 = 32768 bins
    R = np.floor(R / 8).astype(int)
    G = np.floor(G / 8).astype(int)
    B = np.floor(B / 8).astype(int)

    # Indicizzazione nella matrice w2c (32×32×32 → 11)
    index = R + 32*G + 32*32*B
    color_names = w2c[index]  # shape: (H, W, 11)
```

**w2c matrix**: Pre-computed experimentally by [van de Weijer et al., 2009], maps quantized RGB combinations to probability distributions over the 11 color names.

#### 3.4.2 Saliency Map Generation

For each color-name channel (11 channels total):

1. **Multi-threshold binarization**: Apply 16 uniformly spaced thresholds in [0, 255]
2. **Morphological operations**: Closing to fill holes in regions
3. **Border filtering**: Remove contours that touch the image margins
4. **Accumulation**: Sum binary maps over all thresholds and channels

```python
final_saliency = sum(channel_saliency) / (11 * 16)  # normalized average
```

**Output**: Continuous saliency map in [0, 1], where high values indicate regions whose chromatic distribution is distinct from the background.


In [ ]:
# checking edible vs non-edible distribution in sets
edible_counts_train = train_df["edible"].value_counts()
edible_counts_val = val_df["edible"].value_counts()
edible_counts_test = test_df["edible"].value_counts()

# percentages
tmp = edible_counts_train.get(True, 0) / len(train_df) * 100
print(f"In train set: {tmp:.2f}% edible and {100 - tmp:.2f}% non-edible")
tmp = edible_counts_val.get(True, 0) / len(val_df) * 100
print(f"In validation set: {tmp:.2f}% edible and {100 - tmp:.2f}% non-edible")
tmp = edible_counts_test.get(True, 0) / len(test_df) * 100
print(f"In test set: {tmp:.2f}% edible and {100 - tmp:.2f}% non-edible")

# show plot of edible vs non-edible in sets
labels = ["Non-Edible", "Edible"]
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)
axes[0].bar(
    x,
    [edible_counts_train.get(False, 0), edible_counts_train.get(True, 0)],
    color=["#d62728", "#2ca02c"],
)
axes[0].set_title(f"Train (n={len(train_df)})")
axes[0].set_ylabel("Count")
axes[1].bar(
    x,
    [edible_counts_val.get(False, 0), edible_counts_val.get(True, 0)],
    color=["#d62728", "#2ca02c"],
)
axes[1].set_title(f"Validation (n={len(val_df)})")
axes[1].set_ylabel("Count")
axes[2].bar(
    x,
    [edible_counts_test.get(False, 0), edible_counts_test.get(True, 0)],
    color=["#d62728", "#2ca02c"],
)
axes[2].set_title(f"Test (n={len(test_df)})")
axes[2].set_ylabel("Count")
plt.xticks(x, labels)
plt.show()


### 3.5 Comparative Analysis of Segmentation Methods and Selection Rationale

During the exploration phase, we implemented and qualitatively evaluated five segmentation techniques, as outlined in Section 3.1. The **Color Names Saliency** method [van de Weijer et al., 2009]—the most advanced approach among those tested—was particularly promising. This technique maps each RGB pixel to an 11-dimensional probability vector representing basic color names (red, green, blue, etc.), followed by multi-threshold binarization (16 thresholds per channel), morphological closing, border filtering, and accumulation across all 11 channels to produce a normalized saliency map in [0, 1]. 

Theoretically, this method promised superior object highlighting compared to baseline techniques, leveraging perceptual color representations and multi-scale processing (hierarchical SLIC at Z1/Z2/Z3 levels, combined with global/local contrast measures \(U_H\), \(U_G\), \(U_L\)). However, in practice on our mushroom dataset:

- Saliency maps failed to consistently isolate the mushroom cap/gills/stem,  
- Activation persisted over large background areas (soil, leaves, grass),  
- Resulting masks were too noisy/fragmented to produce usable segmented images for CNN training (high false positives in foreground detection).  

**Visual inspection** (e.g., `SCz1`, `SCz2`, `SCz3` maps) confirmed poor discrimination between mushroom and complex natural backgrounds, unlike GrabCut's graph-cut optimization. Consequently, **Color Names Saliency was discarded** from the final pipeline, despite its theoretical appeal. The dataset for Experiment 2 was generated exclusively using the enhanced GrabCut implementation (Section 4.2), which provided the best balance of accuracy, robustness, and computational efficiency. 

This decision highlights a key lesson: while literature promises general improvements, empirical validation on domain-specific data (e.g., natural mushroom images with variable lighting/occlusion) is essential. GrabCut outperformed all alternatives in producing clean foreground isolations suitable for transfer learning.


## 4. GrabCut Algorithm and Advanced Techniques

### 4.1 GrabCut: Foreground-Background Segmentation

**GrabCut** [Rother et al., 2004] is an iterative segmentation algorithm based on Graph Cut and Gaussian Mixture Models (GMM).

#### 4.1.1 Operating Principle

1. **Initialization**: The user specifies a rectangle containing the object of interest
2. **Modeling**: Two GMMs model the chromatic distributions of foreground and background
3. **Graph Cut**: Global optimization via min-cut/max-flow on a graph
4. **Iteration**: Iterative refinement of the GMMs and the segmentation

#### 4.1.2 Implementation with Automatic Initialization

```python
def apply_grabcut(img, cropped_pixels=10, iters=5, mask=None):
    h, w = img.shape[:2]

    # Definizione rettangolo iniziale (crop dei bordi)
    rect = (cropped_pixels, cropped_pixels, 
            w - 2*cropped_pixels, h - 2*cropped_pixels)

    # Inizializzazione maschere GMM
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    # Esecuzione GrabCut
    cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 
                iters, cv2.GC_INIT_WITH_RECT)
```

**cropped_pixels parameter**: Defines how much to “tighten” the initial rectangle. Small values (5–15 pixels) work well for centered images.

**Automatic inversion handling**: If the resulting mask has more background pixels than foreground pixels, it is automatically inverted (heuristic to correct incorrect segmentations).


In [ ]:
# defining a function to return a random image from a given dataframe
def get_random_image(df: pd.DataFrame, img_path_col: str = "image_path") -> UIntArray:
    img_path = df[img_path_col].sample().values[0]
    img = cv2.imread(img_path, cv2.IMREAD_COLOR_RGB)
    if img is None:
        raise ValueError("Image not found or unable to read.")

    img = img.astype(np.uint8, copy=False)
    return img


# correcting paths for reduced dataframe
train_df["image_path"] = train_df["image_path"].apply(lambda p: os.path.join(path, p))

samples = 6

images = []
for _ in range(samples):
    img = get_random_image(train_df)
    images.append(img)

plot_multiple_images(images, size=(3, 3), max_per_row=3, tight_layout=True)

### 4.2 Advanced GrabCut with Guided Initialization

Improved version of GrabCut that uses preliminary analyses to initialize the mask more accurately.

#### 4.2.1 Preprocessing Pipeline

1. **Bilateral Filter**: Smoothing while preserving edges
   ```python
   img_bil = cv2.bilateralFilter(img, d=15, sigmaColor=50, sigmaSpace=75)
   ```

2. **SLIC + Normalized Cuts**: Coarse segmentation into ~100 superpixels
   ```python
   superpixel_mask = slic(img_bil, n_segments=100, compactness=10)
   rag = rag_mean_color(img_bil, superpixel_mask)
   superpixel_mask = cut_normalized(superpixel_mask, rag)
   ```

3. **Otsu Thresholding**: Binary mask based on automatic thresholding

#### 4.2.2 Superpixel Scoring

Each superpixel receives a composite score estimating the probability of belonging to the foreground:

```python
score = 0.20*otsu_overlap + 0.35*color_dist_bg + 0.15*centrality + 0.30*compactness
```

Where:
- **otsu_overlap**: Overlap with the Otsu mask
- **color_dist_bg**: Chromatic distance from the average border color
- **centrality**: Normalized distance from the image center (1 - dist/max_dist)
- **compactness**: Compactness measure (4πA/P²)

#### 4.2.3 GrabCut Mask Initialization

The top-k superpixels with the highest scores are marked as **definite foreground** (GC_FGD), providing GrabCut with a more informed initialization than a simple rectangle.

**Advantage**: Reduction of false positives in images with complex backgrounds or off-centered objects.


In [4]:
def get_image_sizes(img_path: str) -> dict[str, int] | None:
    try:
        img = cv2.imread(img_path, cv2.IMREAD_COLOR_RGB)
        if img is None:
            raise ValueError("Image not found or unable to read.")

        h, w = img.shape[:2]
        return {"width": w, "height": h}
    except Exception as e:
        print(f"Error with {img_path}: {e}")
        return None


if os.path.exists("image_sizes.csv"):
    print("image_sizes.csv already exists. Not overwriting.")
    unique_sizes = pd.read_csv("image_sizes.csv", index_col=0)
else:
    image_paths = train_df["image_path"].tolist()

    # parallel processing to get image sizes for training set images
    with ProcessPoolExecutor(max_workers=8) as executor:
        sizes_list = list(executor.map(get_image_sizes, image_paths))

    sizes = pd.DataFrame(sizes_list)

    # getting 'width - height' combinations
    sizes["width_height"] = (
        sizes["width"].astype(str) + " x " + sizes["height"].astype(str)
    )
    unique_sizes = (
        sizes["width_height"]
        .value_counts()
        .rename("count")  # name of the column
        .to_frame()  # convert Series to DataFrame
    )
    # saving unique sizes to a CSV file
    unique_sizes.to_csv("image_sizes.csv", index=True)

print("Unique sizes: ", len(unique_sizes))
print("Any missing sizes? ", unique_sizes.isnull().any().any())

# calculating 'other' row
top_n = 30
top_sizes = unique_sizes["count"].head(top_n)
other_sizes_count = unique_sizes["count"].iloc[top_n:].sum()

other_series = pd.Series({"Others": other_sizes_count})
top_sizes = pd.concat([top_sizes, other_series])

# plotting the distribution of image sizes
fig, ax = plt.subplots(figsize=(12, 5))
top_sizes.plot(kind="bar", ax=ax, color="#1f77b4")
ax.set_title(f"Top {top_n} dimensions + other")
ax.set_xlabel("Dimensions (width x height)")
ax.set_ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
ax.set_yscale("log")
ax.set_ylim(bottom=1)
plt.tight_layout()
plt.show()

NameError: name 'train_df' is not defined

### 4.3 Final Segmentation Pipeline: Methods Used for Dataset Generation

**Clarification on segmentation methods employed**: While Sections 3.1–3.4 detail the **exploratory evaluation** of multiple techniques (GrabCut, Mean Shift + GrabCut, SLIC + GrabCut, Normalized Cuts + GrabCut, Color Names Saliency), **only the GrabCut-based segmentation was adopted in the final pipeline** for generating the segmented dataset used in Experiment 2. [file:12]

All other methods—despite successful implementation and visual testing on random samples (see Section 4.1.2 and comparative plots)—were evaluated but **discarded** due to insufficient foreground isolation quality, higher computational cost, or inconsistent performance across the dataset variability. Specifically:

- **Color Names Saliency** (Section 3.4): produced overly diffuse maps unsuitable for clean object extraction.  
- **SLIC/Normalized Cuts variants**: effective pre-processing but required GrabCut refinement to yield usable masks.  
- **Mean Shift**: too slow for large-scale processing (~9000 images).  

The **final dataset segmentation** (stored as `*_segmentedpaths.csv` with `grabcutpath` column) was generated via `ThreadPoolExecutor`-parallelized GrabCut processing (Section 5.1), using the **enhanced initialization** from Section 4.2 (bilateral filter + superpixel pre-segmentation + Otsu-guided mask). This produced the `train_loader_2`, `val_loader_2`, `test_loader_2` dataloaders for the second experiment, ensuring clean mushroom isolation while preserving morphological details critical for edibility classification. [file:12]

This choice balances empirical performance with scalability, as confirmed by the generated segmented images (suffix `_grabcut.jpg`).


## 5. Segmented Dataset Generation

### 5.1 Large-Scale GrabCut Application

#### 5.1.1 Parallel Processing with ThreadPoolExecutor

To process the entire dataset (~9000 images) within reasonable time, thread-level parallelization is used:

```python
def grabcut_save_from_df(df, path_col, save_path, iters=2, cropped_pixels=10):
    workers = min(32, (os.cpu_count() or 4) + 4)

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(process_one_image, image_paths))
```

**Rationale for threads**: GrabCut is more I/O-bound (image read/write) than CPU-bound, so threads are preferable to processes to avoid serialization overhead.

#### 5.1.2 Error Handling and Robustness

Each image is processed in an isolated try-except block:
- Read errors → image skipped
- GrabCut failure (foreground too small) → original image preserved with a warning

#### 5.1.3 Result Tracking

For each subset (train/val/test):
1. Creation of a new CSV with an added `grabcut_path` column
2. Saving segmented images with the `_grabcut.jpg` suffix
3. Completeness check before proceeding (`check_csv_done` function)

**Output**:
- `train_segmented_paths.csv`
- `val_segmented_paths.csv`
- `test_segmented_paths.csv`

These CSVs make it easy to load both original and segmented images in subsequent DataLoaders.


## 6. Model Architecture and Training

### 6.1 Preparing the DataLoaders

#### 6.1.1 Custom Dataset Class

```python
class MushroomDataset(Dataset):
    def __init__(self, df, img_path_col, transform=None):
        self.image_paths = df[img_path_col].tolist()
        self.labels = df["edible"].astype(int).tolist()
        self.transform = transform

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]
```

#### 6.1.2 Applied Transformations

**Training set** (with data augmentation):
```python
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])
```

**Validation/Test set** (without augmentation):
```python
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])
```

**Normalization**: Mean and standard deviation values are those of ImageNet, required because the model is pre-trained on this dataset.

#### 6.1.3 DataLoader Configuration

```python
train_loader = DataLoader(train_dataset, batch_size=32, 
                         shuffle=True, num_workers=4)
```

- **batch_size=32**: Compromise between speed and GPU memory usage
- **shuffle=True**: Randomizes sample order to reduce overfitting
- **num_workers**: Parallel data loading (I/O-bound)


In [5]:
image_size = 224
pil = Image.fromarray(img)

transform = transforms.Compose(
    [transforms.Resize(image_size), transforms.CenterCrop(image_size)]
)
img = transform(pil)
img = np.array(img)

fig = plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("Resized and Center Cropped Image")
plt.axis("off")
plt.show()


def generate_hierarchy(image):
    h, w = image.shape[:2]

    z1 = slic(image, n_segments=400, compactness=5)

    image_z2 = cv2.resize(
        image, (int(w * 2 / 3), int(h * 2 / 3)), interpolation=cv2.INTER_LINEAR
    )
    z2 = slic(image_z2, n_segments=200, compactness=5)

    image_z3 = cv2.resize(
        image, (int(w / 2), int(h / 2)), interpolation=cv2.INTER_LINEAR
    )
    z3 = slic(image_z3, n_segments=100, compactness=5)

    return (image, z1), (image_z2, z2), (image_z3, z3)


# each pixel has a value corresponding to its segment [0, num_segments-1]
(z1_img, segment_map_z1), (z2_img, segment_map_z2), (z3_img, segment_map_z3) = (
    generate_hierarchy(img)
)

# calculating average color for each segment in z1, z2, and z3 for CieLab color space
def get_avg_color_per_segment(img, segment_map):
    # to CieLab color space
    img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    avg_color_dict = {}
    for seg_id in np.unique(segment_map):
        # for each segment number, we create a mask with pixels belonging to it
        mask = segment_map == seg_id
        # calculate average color for each channel
        avg_color = img[mask].mean(axis=0)
        avg_color_dict[seg_id] = avg_color
    # shape of avg_color_dict should be (num_segments, 3)
    return avg_color_dict


color_dict_z1 = get_avg_color_per_segment(z1_img, segment_map_z1)
color_dict_z2 = get_avg_color_per_segment(z2_img, segment_map_z2)
color_dict_z3 = get_avg_color_per_segment(z3_img, segment_map_z3)
print("Len of color_dict_z1:", len(color_dict_z1))
print("Len of color_dict_z2:", len(color_dict_z2))
print("Len of color_dict_z3:", len(color_dict_z3))

# plotting the segmented images with average colors
segmented_img_z1 = np.zeros_like(z1_img)
for seg_id in np.unique(segment_map_z1):
    mask = segment_map_z1 == seg_id
    segmented_img_z1[mask] = color_dict_z1[seg_id]
segmented_img_z2 = np.zeros_like(z2_img)
for seg_id in np.unique(segment_map_z2):
    mask = segment_map_z2 == seg_id
    segmented_img_z2[mask] = color_dict_z2[seg_id]
segmented_img_z3 = np.zeros_like(z3_img)
for seg_id in np.unique(segment_map_z3):
    mask = segment_map_z3 == seg_id
    segmented_img_z3[mask] = color_dict_z3[seg_id]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(segmented_img_z1, cv2.COLOR_LAB2RGB))
axes[0].set_title("Segmented Z1")
axes[1].imshow(cv2.cvtColor(segmented_img_z2, cv2.COLOR_LAB2RGB))
axes[1].set_title("Segmented Z2")
axes[2].imshow(cv2.cvtColor(segmented_img_z3, cv2.COLOR_LAB2RGB))
axes[2].set_title("Segmented Z3")
plt.show()

NameError: name 'Image' is not defined

### 6.2 MobileNetV3-Small Architecture

#### 6.2.1 Model Characteristics

**MobileNetV3-Small** [Howard et al., 2019] is a CNN optimized for mobile devices:

- **Total parameters**: ~2.5M (vs 25M for ResNet-50)
- **Operations**: ~56M MACs (Multiply-Accumulate Operations)
- **Key techniques**:
  - Depthwise Separable Convolutions
  - Squeeze-and-Excitation blocks
  - h-swish activation function
  - Network Architecture Search (NAS) for optimization

#### 6.2.2 Modification for Binary Classification

The pre-trained model has an output layer for 1000 ImageNet classes. It is modified to output a single value:

```python
# Layer originale
model.classifier[3] = Linear(in_features=576, out_features=1000)

# Layer modificato per classificazione binaria
model.classifier[3] = Linear(in_features=576, out_features=1)
```

**Rationale**: Single output + BCEWithLogitsLoss is equivalent to two outputs + CrossEntropyLoss, but computationally more efficient.

#### 6.2.3 Transfer Learning

Backbone (feature extractor) weights are initialized from ImageNet:
- **Advantage**: low-level features (edges, textures, shapes) are already learned
- **Strategy**: fine-tuning with differentiated learning rates (backbone: 1e-5, classifier: 2e-4)


In [6]:
# k-means clustering on the pixel colors in CieLab color space

# the number optimal number of clusters can be determined using the elbow method or silhouette score
def kmeans_on_segments(segmentation_map, color_dict):
    # convert color_dict to numpy array
    X = np.array(list(color_dict.values()))

    best_score = -np.inf
    best_km = None
    ks = range(5, min(20, len(color_dict)))
    for k in ks:
        # initialize k-means model and predict labels for each segment
        km = KMeans(n_clusters=k, random_state=42, n_init="auto").fit(X)
        score = silhouette_score(X, km.labels_)
        if score > best_score:
            best_score = score
            best_km = km
    # map of segment to cluster label
    labels = best_km.labels_
    # map of cluster id to centroid color
    centroids = best_km.cluster_centers_
    # we are essentially mapping each segment to its cluster label, reducing the number of colors
    print("Best silhouette score:", best_score)
    print("Labels shape:", labels.shape)
    print("Centroids shape:", centroids.shape)

    # mapping segment ids to cluster colors
    cluster_lab = np.zeros((*segmentation_map.shape, 3), dtype=np.float64)
    seg_ids = list(color_dict.keys())
    for idx, seg_id in enumerate(seg_ids):
        cluster_id = labels[idx]
        cluster_lab[segmentation_map == seg_id] = centroids[cluster_id]
    cluster_rgb = cv2.cvtColor(cluster_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    return labels, centroids, cluster_rgb


labels_z1, centroids_z1, rgb_map_z1 = kmeans_on_segments(segment_map_z1, color_dict_z1)
labels_z2, centroids_z2, rgb_map_z2 = kmeans_on_segments(segment_map_z2, color_dict_z2)
labels_z3, centroids_z3, rgb_map_z3 = kmeans_on_segments(segment_map_z3, color_dict_z3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb_map_z1)
axes[0].set_title(f"K-Means Clustering Z1 ({len(centroids_z1)} clusters)")
axes[1].imshow(rgb_map_z2)
axes[1].set_title(f"K-Means Clustering Z2 ({len(centroids_z2)} clusters)")
axes[2].imshow(rgb_map_z3)
axes[2].set_title(f"K-Means Clustering Z3 ({len(centroids_z3)} clusters)")
plt.show()

NameError: name 'segment_map_z1' is not defined

### 6.3 Training Configuration

#### 6.3.1 Loss Function with Class Weighting

To handle potential residual class imbalance:

```python
edible_counts = train_df["edible"].value_counts()
pos_weight = edible_counts[False] / edible_counts[True]

if pos_weight != 1:
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
else:
    criterion = nn.BCEWithLogitsLoss()
```

**Binary Cross-Entropy with Logits**:
```
Loss = -[y * log(σ(x)) + (1-y) * log(1-σ(x))]
```
With pos_weight w:
```
Loss = -[y * w * log(σ(x)) + (1-y) * log(1-σ(x))]
```

This penalizes errors on the less represented class more heavily.

#### 6.3.2 Optimizer with Differentiated Learning Rates

```python
param_groups = [
    {"params": model.features.parameters(), "lr": 1e-5},  # backbone
    {"params": model.classifier.parameters(), "lr": 2e-4} # head
]
optimizer = optim.Adam(param_groups)
```

**Rationale**: The pre-trained backbone requires fine adjustments, while the randomly initialized classifier needs more aggressive updates.

#### 6.3.3 Learning Rate Scheduling

```python
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)
```

Reduces the learning rate by 50% if the validation loss does not improve for 2 consecutive epochs, preventing oscillations and facilitating convergence.


In [7]:
# calculating the area of each cluster in each segmentation level
def calculate_cluster_areas(segmentation_map, labels):
    area_dict = {}
    seg_ids = list(np.unique(segmentation_map))
    for idx, seg_id in enumerate(seg_ids):
        cluster_id = labels[idx]
        area = np.sum(segmentation_map == seg_id)
        if cluster_id in area_dict:
            area_dict[cluster_id] += area
        else:
            area_dict[cluster_id] = area
    return area_dict


area_z1 = calculate_cluster_areas(segment_map_z1, labels_z1)
area_z2 = calculate_cluster_areas(segment_map_z2, labels_z2)
area_z3 = calculate_cluster_areas(segment_map_z3, labels_z3)


# calculating number of pixels touching the image border for each cluster
def calculate_border_areas(segmentation_map, labels):
    border_dict = {}
    h, w = segmentation_map.shape
    seg_ids = list(np.unique(segmentation_map))
    border_pixels = np.concatenate(
        [
            segmentation_map[0, :],  # top row
            segmentation_map[h - 1, :],  # bottom row
            segmentation_map[:, 0],  # left column
            segmentation_map[:, w - 1],  # right column
        ]
    )
    for idx, seg_id in enumerate(seg_ids):
        cluster_id = labels[idx]
        border_area = np.sum(border_pixels == seg_id)
        if cluster_id in border_dict:
            border_dict[cluster_id] += border_area
        else:
            border_dict[cluster_id] = border_area
    return border_dict


border_area_z1 = calculate_border_areas(segment_map_z1, labels_z1)
border_area_z2 = calculate_border_areas(segment_map_z2, labels_z2)
border_area_z3 = calculate_border_areas(segment_map_z3, labels_z3)


# calculating the spatial compactness for each cluster
# we are calculating the euclidean distance of each pixel in the segment to the cluster to the cluster center
def calculate_spatial_compactness(segmentation_map, labels):
    unique_clusters = np.unique(labels)
    compactness_dict = {cluster_id: 0.0 for cluster_id in unique_clusters}
    seg_ids = list(np.unique(segmentation_map))
    # getting the list of segments in the cluster and creating a mask for each cluster
    centers = {}
    for cluster_id in unique_clusters:
        segs_in_cluster = [
            seg_id for idx, seg_id in enumerate(seg_ids) if labels[idx] == cluster_id
        ]
        mask = np.isin(segmentation_map, segs_in_cluster)
        ys, xs = np.where(mask)
        x_center = np.mean(xs)
        y_center = np.mean(ys)
        centers[cluster_id] = (x_center, y_center)

    for idx, seg_id in enumerate(seg_ids):
        # getting cluster center
        cluster_id = labels[idx]
        x_cc, y_cc = centers[cluster_id][0], centers[cluster_id][1]
        # getting all pixel coordinates in the segment
        ys, xs = np.where(segmentation_map == seg_id)
        # calculating centroid of the segment
        x_center = np.mean(xs)
        y_center = np.mean(ys)
        # calculating distance of centroid of segment to cluster centroid
        distance = np.sqrt((x_center - x_cc) ** 2 + (y_center - y_cc) ** 2)
        compactness_dict[cluster_id] += distance
    return compactness_dict


compactness_z1 = calculate_spatial_compactness(segment_map_z1, labels_z1)
compactness_z2 = calculate_spatial_compactness(segment_map_z2, labels_z2)
compactness_z3 = calculate_spatial_compactness(segment_map_z3, labels_z3)


# calculating the background probability based on these features
def calculate_background_probability(area_dict, border_area_dict, compactness_dict):
    # calculating BndCon for each cluster
    bnd_con_values = {}
    for cluster_id in area_dict.keys():
        area = area_dict.get(cluster_id, 0)
        border_area = border_area_dict.get(cluster_id, 0)
        compactness = compactness_dict.get(cluster_id, 0)

        if area <= 0:
            bnd_con_values[cluster_id] = 0.0
        else:
            val = (border_area / sqrt(area)) * compactness
            bnd_con_values[cluster_id] = val

    # calculating mean BndCon across all clusters
    all_values = list(bnd_con_values.values())
    mean_bnd_con = np.mean(all_values)
    # avoiding division by zero
    denominator = mean_bnd_con if mean_bnd_con > 1e-6 else 1.0
    denominator = 2 * (denominator**2)

    # calculating background probability for each cluster
    prob_dict = {}
    for cluster_id, bnd_con in bnd_con_values.items():
        prob = 1 - np.exp(-bnd_con / denominator)
        prob_dict[cluster_id] = prob

    return prob_dict


bg_prob_z1 = calculate_background_probability(area_z1, border_area_z1, compactness_z1)
bg_prob_z2 = calculate_background_probability(area_z2, border_area_z2, compactness_z2)
bg_prob_z3 = calculate_background_probability(area_z3, border_area_z3, compactness_z3)


# generating binary mask, background when probability is higher than threshold * avg_bg_prob
def calculate_binary_mask(segmentation_map, labels, bg_prob_dict, threshold=0.5):
    binary_mask = np.zeros(segmentation_map.shape, dtype="uint8")
    avg_bg_prob = np.mean(list(bg_prob_dict.values()))
    seg_ids = list(np.unique(segmentation_map))
    for idx, seg_id in enumerate(seg_ids):
        cluster_id = labels[idx]
        prob = bg_prob_dict.get(cluster_id)
        if prob >= threshold * avg_bg_prob:
            binary_mask[segmentation_map == seg_id] = 0  # background
        else:
            binary_mask[segmentation_map == seg_id] = 1  # foreground

    return binary_mask


SA_z1 = calculate_binary_mask(segment_map_z1, labels_z1, bg_prob_z1, threshold=1)
SA_z2 = calculate_binary_mask(segment_map_z2, labels_z2, bg_prob_z2, threshold=1)
SA_z3 = calculate_binary_mask(segment_map_z3, labels_z3, bg_prob_z3, threshold=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(SA_z1, cmap="gray")
axes[0].set_title("Boundary Saliency Mask Z1")
axes[1].imshow(SA_z2, cmap="gray")
axes[1].set_title("Boundary Saliency Mask Z2")
axes[2].imshow(SA_z3, cmap="gray")
axes[2].set_title("Boundary Saliency Mask Z3")
plt.show()

NameError: name 'segment_map_z1' is not defined

### 6.4 Training and Evaluation Loop

#### 6.4.1 Evaluation Function

```python
def evaluate_model(model, dataloader, device, criterion):
    model.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs.to(device))
            loss = criterion(outputs, labels.float().unsqueeze(1).to(device))
            preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy()
```

**Computed metrics**:
- Accuracy, Precision, Recall, F1-Score
- Confusion Matrix: [[TN, FP], [FN, TP]]

#### 6.4.2 Full Training Loop

```python
def train_and_evaluate_model(...):
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs.to(device))
            loss = criterion(outputs, labels.to(device))
            loss.backward()
            optimizer.step()

        # Validation phase
        val_loss, acc, prec, rec, f1, cm = evaluate_model(...)
        scheduler.step(val_loss)

        # Checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_model_type{model_type}.pth")
```

#### 6.4.3 Implicit Early Stopping

The model with the best validation loss is automatically saved. At the end of training, this checkpoint is loaded for the final evaluation on the test set, avoiding overfitting on the last epochs.


In [8]:
# Corse saliency map generation based on color priors

# calculating color histogram in high-dimensional color space
# 8 dimensions: RGB, CieLab, hue, saturation
def get_hd_image(img):
    rgb = img
    rgb = rgb / 255
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = lab[:, :, 0] / 255
    lab[:, :, 1] = (lab[:, :, 1] + 128) / 255
    lab[:, :, 2] = (lab[:, :, 2] + 128) / 255
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hsv[:, :, 0] = hsv[:, :, 0] / 179.0
    hsv[:, :, 1] = hsv[:, :, 1] / 255.0
    # shape (H, W, 8)
    hd_img = np.concatenate([rgb, lab, hsv[:, :, 0:2]], axis=2)

    return hd_img


hd_z1 = get_hd_image(z1_img)
hd_z2 = get_hd_image(z2_img)
hd_z3 = get_hd_image(z3_img)


def calculate_hd_avg_color_per_segment(img, segment_map):
    avg_color_dict = {}
    for seg_id in np.unique(segment_map):
        mask = segment_map == seg_id
        avg_color = img[mask].mean(axis=0)
        avg_color_dict[seg_id] = avg_color
    return avg_color_dict


# shape of hd_color_dict should be (num_segments, 8)
hd_color_dict_z1 = calculate_hd_avg_color_per_segment(hd_z1, segment_map_z1)
hd_color_dict_z2 = calculate_hd_avg_color_per_segment(hd_z2, segment_map_z2)
hd_color_dict_z3 = calculate_hd_avg_color_per_segment(hd_z3, segment_map_z3)


# calculate 8 bins histogram for each channel in hd color space
# CORREZIONE
def calculate_histogram(img, img_seg):
    histograms = []
    bins = 8
    seg_ids = np.unique(img_seg)

    for seg_id in seg_ids:
        mask = img_seg == seg_id
        pixels = img[mask]

        # calculating histogram for each of the 8 channels
        channel_hists = []
        for ch in range(pixels.shape[1]):
            hist_ch, _ = np.histogram(pixels[:, ch], bins=bins, range=(0, 256))
            channel_hists.append(hist_ch)

        full_hist = np.concatenate(channel_hists)
        full_hist = full_hist.astype("float") / (full_hist.sum() + 1e-6)

        histograms.append(full_hist)

    return np.array(histograms)


hd_hist_z1 = calculate_histogram(hd_z1, segment_map_z1)
hd_hist_z2 = calculate_histogram(hd_z2, segment_map_z2)
hd_hist_z3 = calculate_histogram(hd_z3, segment_map_z3)


# calculate chi-square distance between histograms
def calculate_chi_square_distance(histograms):
    num_segments = histograms.shape[0]
    U_H = np.zeros(num_segments)

    for i in range(num_segments):
        chi_sq_sum = 0.0
        for j in range(num_segments):
            if i == j:
                continue

            hist_i = histograms[i]
            hist_j = histograms[j]

            chi_sq_sum += np.sum((hist_i - hist_j) ** 2 / (hist_i + hist_j + 1e-6))

        U_H[i] = chi_sq_sum
    return U_H


U_H_z1 = calculate_chi_square_distance(hd_hist_z1)
U_H_z2 = calculate_chi_square_distance(hd_hist_z2)
U_H_z3 = calculate_chi_square_distance(hd_hist_z3)


# calculate global contrast against other segments
def calculate_global_contrast(color_dict):
    seg_ids = list(color_dict.keys())
    num_segments = len(seg_ids)
    U_G = np.zeros(num_segments)

    for i in range(num_segments):
        color_i = color_dict[seg_ids[i]]
        contrast_sum = 0.0
        for j in range(num_segments):
            if i == j:
                continue
            color_j = color_dict[seg_ids[j]]
            # euclidean distance
            contrast_sum += np.linalg.norm(color_i - color_j)
        U_G[i] = contrast_sum
    return U_G


U_G_z1 = calculate_global_contrast(color_dict_z1)
U_G_z2 = calculate_global_contrast(color_dict_z2)
U_G_z3 = calculate_global_contrast(color_dict_z3)


# calculating local contrast
def calculate_local_contrast(color_dict, segment_map):
    seg_ids = list(color_dict.keys())
    num_segments = len(seg_ids)
    U_L = np.zeros(num_segments)

    H = segment_map.shape[0]
    W = segment_map.shape[1]

    centroids = {}
    for seg_id in seg_ids:
        ys, xs = np.where(segment_map == seg_id)
        x_center = np.mean(xs) / W
        y_center = np.mean(ys) / H
        centroids[seg_id] = (x_center, y_center)

    matrix = np.zeros((num_segments, num_segments))

    for i in range(num_segments):
        for j in range(num_segments):
            if i == j:
                continue
            # euclidean distance
            centroid_i = centroids[seg_ids[i]]
            centroid_j = centroids[seg_ids[j]]
            spatial_dist = (
                np.linalg.norm(np.array(centroid_i) - np.array(centroid_j))
            ) ** 2
            raw_weight = np.exp(-spatial_dist / 0.5)
            matrix[i, j] = raw_weight
    # normalizing weights
    # summing each column so we get the sum of the raw weights for each segment i
    E_factor = matrix.sum(axis=1) + 1e-6
    matrix = matrix / E_factor[:, np.newaxis]
    for i in range(num_segments):
        contrast_sum = 0.0
        for j in range(num_segments):
            if i == j:
                continue
            color_j = color_dict[seg_ids[j]]
            dist = np.linalg.norm(color_dict[seg_ids[i]] - color_j)
            weight = matrix[i, j]
            contrast_sum += dist * weight
        U_L[i] = contrast_sum

    return U_L


U_L_z1 = calculate_local_contrast(color_dict_z1, segment_map_z1)
U_L_z2 = calculate_local_contrast(color_dict_z2, segment_map_z2)
U_L_z3 = calculate_local_contrast(color_dict_z3, segment_map_z3)


# Normalizing and combining the three contrast measures to get the final saliency map
def normalize_and_combine(U_H, U_G, U_L):
    U_H_norm = (U_H - U_H.min()) / (U_H.max() - U_H.min() + 1e-6)
    U_G_norm = (U_G - U_G.min()) / (U_G.max() - U_G.min() + 1e-6)
    U_L_norm = (U_L - U_L.min()) / (U_L.max() - U_L.min() + 1e-6)

    S = U_H_norm + U_G_norm + U_L_norm
    return S


S_z1 = normalize_and_combine(U_H_z1, U_G_z1, U_L_z1)
S_z2 = normalize_and_combine(U_H_z2, U_G_z2, U_L_z2)
S_z3 = normalize_and_combine(U_H_z3, U_G_z3, U_L_z3)


def generate_saliency_map(segment_map, S):
    h, w = segment_map.shape
    saliency_map = np.zeros((h, w), dtype="float32")
    seg_ids = list(np.unique(segment_map))
    for idx, seg_id in enumerate(seg_ids):
        saliency_map[segment_map == seg_id] = S[idx]
    return saliency_map


# generating saliency maps
saliency_map_z1 = generate_saliency_map(segment_map_z1, S_z1)
saliency_map_z2 = generate_saliency_map(segment_map_z2, S_z2)
saliency_map_z3 = generate_saliency_map(segment_map_z3, S_z3)


# each image will be divided in 2x2, 3x3 and 4x4 grids respectively
# we calculate Otsu's threshold with 8 classes for each grid cell
# we will have for each segment a value between 0 and 7
def get_recursive_thresholds(values, depth=3):
    # if depth is 0 or values are uniform, return empty list
    if depth == 0 or len(values) == 0 or np.min(values) == np.max(values):
        return []

    thresh = threshold_otsu(values)

    lower_mask = values <= thresh
    upper_mask = values > thresh

    lower_thresholds = get_recursive_thresholds(values[lower_mask], depth - 1)
    upper_thresholds = get_recursive_thresholds(values[upper_mask], depth - 1)

    return sorted(lower_thresholds + [thresh] + upper_thresholds)


def calculate_adaptive_thresholds(segment_map, saliency_map, grid_size):
    H, W = segment_map.shape
    cell_h = H // grid_size
    cell_w = W // grid_size

    seg_ids = list(np.unique(segment_map))
    threshold_dict = {seg_id: [] for seg_id in seg_ids}

    for i in range(grid_size):
        for j in range(grid_size):
            # Getting the cell boundaries
            y_start, y_end = i * cell_h, (i + 1) * cell_h
            x_start, x_end = j * cell_w, (j + 1) * cell_w
            if i == grid_size - 1:
                y_end = H
            if j == grid_size - 1:
                x_end = W

            cell_saliency = saliency_map[y_start:y_end, x_start:x_end]
            cell_segments = segment_map[y_start:y_end, x_start:x_end]

            # calculating multi-level Otsu threshold is too complex on 256 levels
            # we can quantize the saliency values to x levels or iterate on binary Otsu

            cell_saliency_quantized = (cell_saliency * 255).astype(np.uint8)
            thresholds = get_recursive_thresholds(
                cell_saliency_quantized.flatten(), depth=3
            )
            scores = np.digitize(cell_saliency_quantized, bins=thresholds)

            for seg_id in np.unique(cell_segments):
                mask = cell_segments == seg_id
                avg_score = np.mean(scores[mask])
                threshold_dict[seg_id].append(avg_score)

    # if a segment is in different grid cells we average the thresholds
    # the edge case is if theres only one pixel of a segment in that grid cell
    # we could consider only the cell where theres the centroid but for now we average all
    avg_threshold_dict = {}
    for seg_id, scores in threshold_dict.items():
        avg_threshold_dict[seg_id] = np.mean(scores)

    return avg_threshold_dict


thresh_dict_z1 = (
    calculate_adaptive_thresholds(segment_map_z1, saliency_map_z1, grid_size=2),
    calculate_adaptive_thresholds(segment_map_z1, saliency_map_z1, grid_size=3),
    calculate_adaptive_thresholds(segment_map_z1, saliency_map_z1, grid_size=4),
)
thresh_dict_z2 = (
    calculate_adaptive_thresholds(segment_map_z2, saliency_map_z2, grid_size=2),
    calculate_adaptive_thresholds(segment_map_z2, saliency_map_z2, grid_size=3),
    calculate_adaptive_thresholds(segment_map_z2, saliency_map_z2, grid_size=4),
)
thresh_dict_z3 = (
    calculate_adaptive_thresholds(segment_map_z3, saliency_map_z3, grid_size=2),
    calculate_adaptive_thresholds(segment_map_z3, saliency_map_z3, grid_size=3),
    calculate_adaptive_thresholds(segment_map_z3, saliency_map_z3, grid_size=4),
)


# summing thresholds from different grid sizes
def combine_thresholds(segmentation_map, g2, g3, g4):
    combined_thresh = {}
    seg_ids = list(np.unique(segmentation_map))
    for seg_id in seg_ids:
        t2 = g2.get(seg_id, 0)
        t3 = g3.get(seg_id, 0)
        t4 = g4.get(seg_id, 0)
        combined_thresh[seg_id] = t2 + t3 + t4
    return combined_thresh


combined_thresh_z1 = combine_thresholds(segment_map_z1, *(thresh_dict_z1))
combined_thresh_z2 = combine_thresholds(segment_map_z2, *(thresh_dict_z2))
combined_thresh_z3 = combine_thresholds(segment_map_z3, *(thresh_dict_z3))


def generate_final_color_saliency_mask(segment_map, combined_thresh):
    h, w = segment_map.shape
    saliency_mask = np.zeros((h, w), dtype="float32")
    seg_ids = list(np.unique(segment_map))
    for seg_id in seg_ids:
        thresh = combined_thresh.get(seg_id)
        mask = segment_map == seg_id
        if thresh >= 18:
            saliency_mask[mask] = 1  # foreground
        elif thresh <= 6:
            saliency_mask[mask] = 0  # background
        else:
            saliency_mask[mask] = 0.5  # uncertain
    return saliency_mask


SB_z1 = generate_final_color_saliency_mask(segment_map_z1, combined_thresh_z1)
SB_z2 = generate_final_color_saliency_mask(segment_map_z2, combined_thresh_z2)
SB_z3 = generate_final_color_saliency_mask(segment_map_z3, combined_thresh_z3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(SB_z1, cmap="gray")
axes[0].set_title("Color Saliency Mask Z1")
axes[1].imshow(SB_z2, cmap="gray")
axes[1].set_title("Color Saliency Mask Z2")
axes[2].imshow(SB_z3, cmap="gray")
axes[2].set_title("Color Saliency Mask Z3")
plt.show()


NameError: name 'z1_img' is not defined

### 6.5 Results Visualization

#### 6.5.1 Training Curves

```python
def plot_evaluation(train_loss, val_loss, train_acc, val_acc, 
                   precision, recall, f1):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    # Subplot 1: Loss curves
    # Subplot 2: Accuracy curves  
    # Subplot 3: F1 score
    # Subplot 4: Precision/Recall
```

**Interpretation**:
- **Diverging losses**: overfitting if val_loss increases while train_loss decreases
- **Accuracy plateau**: possible saturation; consider architectural changes
- **F1 vs Accuracy**: F1 is more informative under class imbalance
- **Precision-Recall trade-off**: important to assess model conservativeness

#### 6.5.2 Confusion Matrix

Visualization of the 2×2 matrix:

```
              Predicted
              Non-Edible  Edible
Actual Non-E     TN         FP
       Edible    FN         TP
```

**Critical focus**: Minimizing FP (false positives) is a priority in the mushroom context, where incorrectly classifying a poisonous mushroom as edible can have fatal consequences.


## 7. Experiment 1: Training on Original Images

### 7.1 Configuration and Objectives

This experiment constitutes the **baseline** comparison:
- **Input**: original RGB images without preprocessing
- **Objective**: evaluate the model's capabilities when it has access to the full visual context (habitat, substrate, surrounding elements)

### 7.2 Hypothesis

The model might benefit from:
- **Ecological information**: type of soil, presence of specific trees
- **Seasonal context**: leaves, snow, humidity
- **Scale visual cues**: dimensional references relative to the environment

However, it might suffer from:
- **Background noise**: complex textures that are not relevant
- **Overfitting to the background**: learning spurious environment-species correlations

### 7.3 DataLoader and Training

Use of `train_loader_1`, `val_loader_1`, `test_loader_1`, which load the original images from the paths specified in the `image_path` column of the DataFrames.


In [9]:
def load_w2c_matrix(filepath="w2c.mat"):
    data = scipy.io.loadmat(filepath)
    w2c = data["w2c"]
    return w2c


# converting rgb image in color names
def im2c(img, w2c):
    R = img[:, :, 0]
    G = img[:, :, 1]
    B = img[:, :, 2]

    R = np.floor(R / 8).astype(int)
    G = np.floor(G / 8).astype(int)
    B = np.floor(B / 8).astype(int)

    index = R + 32 * G + 32 * 32 * B
    out = w2c[index]
    out = (out * 255).astype(np.uint8)
    return out


w2c = load_w2c_matrix()
# we now have a (H, W, 11) array where each pixel is represented by an 11-dim color name vector
# the sum over the 11 channels should be 1 for each pixel, each is the probability of that color name
cn_z1 = im2c(z1_img, w2c)
cn_z2 = im2c(z2_img, w2c)
cn_z3 = im2c(z3_img, w2c)


def generate_refined_saliency(cn_img, num_thresholds=16):
    H, W, C = cn_img.shape

    final_saliency = np.zeros((H, W), dtype=np.float32)
    thresholds = np.linspace(0, 255, num_thresholds)

    # kernel to be used for morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    def process_mask(binary_mask):
        # morphological closing
        closed = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
        # find contours to
        contours, _ = cv2.findContours(
            closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        mask_out = np.zeros_like(binary_mask)
        for cnt in contours:
            x, y, cw, ch = cv2.boundingRect(cnt)

            # if the contour touches the border, skip it
            if x == 0 or y == 0 or (x + cw) >= W or (y + ch) >= H:
                continue

            # else we fill the contour
            cv2.drawContours(mask_out, [cnt], -1, 255, -1)

        return mask_out

    for ch in range(C):
        channel = cn_img[:, :, ch]
        channel_accum = np.zeros((H, W), dtype=np.float32)

        for t in thresholds:
            mask_pos = (channel >= t).astype(np.uint8) * 255
            if cv2.countNonZero(mask_pos) > 0:
                processed_pos = process_mask(mask_pos)
                channel_accum += processed_pos

            mask_neg = (channel < t).astype(np.uint8) * 255
            if cv2.countNonZero(mask_neg) > 0:
                processed_neg = process_mask(mask_neg)
                channel_accum += processed_neg

        # we avg the accumulated map for this channel
        channel_avg = channel_accum / (255.0 * 2 * num_thresholds)
        final_saliency += channel_avg

    # final avg over all channels
    final_saliency /= C

    return final_saliency


SC_z1 = generate_refined_saliency(cn_z1)
SC_z2 = generate_refined_saliency(cn_z2)
SC_z3 = generate_refined_saliency(cn_z3)

# Visualizzazione
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(SC_z1, cmap="gray")
axes[0].set_title("Center Saliency Mask Z1")
axes[1].imshow(SC_z2, cmap="gray")
axes[1].set_title("Center Saliency Mask Z2")
axes[2].imshow(SC_z3, cmap="gray")
axes[2].set_title("Center Saliency Mask Z3")
plt.show()


NameError: name 'scipy' is not defined

## 8. Experiment 2: Training on Segmented Images

### 8.1 Configuration and Objectives

This experiment evaluates the impact of background removal:
- **Input**: images segmented via GrabCut (black/removed background)
- **Objective**: verify whether isolating the mushroom improves classification by removing distractors

### 8.2 Hypothesis

**Expected advantages**:
1. **Morphological focus**: the model concentrates on discriminative mushroom features (cap, gills, stem)
2. **Reduced overfitting**: removal of spurious correlations with the background
3. **Generalization**: lower dependence on environmental conditions specific to the training set

**Potential disadvantages**:
1. **Information loss**: habitat can be a useful cue (e.g., mushrooms on wood vs soil)
2. **Segmentation errors**: artifacts introduced by GrabCut may confuse the model
3. **Size bias**: background removal changes relative proportions

### 8.3 DataLoader and Training

Use of `train_loader_2`, `val_loader_2`, `test_loader_2`, which load images from the paths in the `grabcut_path` column generated during segmentation.

### 8.4 Initialization Considerations

The model is **re-initialized** with fresh ImageNet weights (no weights are transferred from Experiment 1) to ensure a fair comparison and avoid biases due to training order.


In [10]:
def normalize_map(m):
    m_norm = (m - m.min()) / (m.max() - m.min() + 1e-6)
    return m_norm


# calculating the quadratic euclidean distance between the maps at the same level
def calculate_fusion_map(SA, SB, SC):
    a = (np.linalg.norm(SA - SB)) ** 2
    b = (np.linalg.norm(SB - SC)) ** 2
    c = (np.linalg.norm(SC - SA)) ** 2

    if min(a, b, c) == a:
        out = SA * SB + 1e-6
    elif min(a, b, c) == b:
        out = SB * SC + 1e-6
    else:
        out = SC * SA + 1e-6
    return normalize_map(out)


diff_z1 = calculate_fusion_map(SA_z1, SB_z1, SC_z1)
diff_z2 = calculate_fusion_map(SA_z2, SB_z2, SC_z2)
diff_z3 = calculate_fusion_map(SA_z3, SB_z3, SC_z3)
# interpolating back to original image size
H, W = img.shape[:2]
diff_z2 = cv2.resize(diff_z2, (W, H), interpolation=cv2.INTER_LINEAR)
diff_z3 = cv2.resize(diff_z3, (W, H), interpolation=cv2.INTER_LINEAR)

a = (np.linalg.norm(diff_z1 - diff_z2)) ** 2
b = (np.linalg.norm(diff_z2 - diff_z3)) ** 2
c = (np.linalg.norm(diff_z3 - diff_z1)) ** 2

if max(a, b, c) == a:
    saliency_map = diff_z1 * diff_z2 + 1e-6
elif max(a, b, c) == b:
    saliency_map = diff_z2 * diff_z3 + 1e-6
else:
    saliency_map = diff_z3 * diff_z1 + 1e-6

saliency_map = normalize_map(saliency_map)
# final saliency map visualization and comparison with original image
# and other saliency methods
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[1].imshow(saliency_map, cmap="gray")
axes[1].set_title("Final Saliency Map")

plt.show()


NameError: name 'SA_z1' is not defined

## 9. Comparative Analysis and Conclusions

### 9.1 Comparison Metrics

After training both experiments, the following metrics are compared:

| Metric | Experiment 1 (Original) | Experiment 2 (Segmented) |
|--------|--------------------------|--------------------------|
| Test Accuracy | - | - |
| Test Precision | - | - |
| Test Recall | - | - |
| Test F1-Score | - | - |
| False Positives | - | - |

*(Values to be filled after execution)*

### 9.2 Final Considerations

**Key trade-off**:
- Contextual information vs noise reduction
- Robustness to environmental variability vs morphological focus

**Research question**: Is there a statistically significant advantage in using segmented images for this task?

**Possible extensions**:
1. Ensemble of the two models (voting or stacking)
2. Segmentation with deep techniques (U-Net, Mask R-CNN)
3. Attention mechanisms for selective focus on relevant regions
4. Multi-task learning: classification + segmentation simultaneously

### 9.3 Limitations and Future Work

**Methodological limitations**:
- Dataset limited to 50 species out of 169 total
- Imperfect segmentation (dependency on GrabCut)
- Lack of advanced data augmentation (mixup, cutout)

**Future directions**:
- Extension to multi-class classification (169 species)
- Integration of metadata (GPS, season, altitude)
- Interpretability via Grad-CAM for discriminative feature localization
- Deployment on mobile devices for in-situ recognition


### Image Processing and Visualization

In this section of the code, a series of image-processing operations are performed, followed by visualization of the results.

#### Conversion to Grayscale and CieLab

- **Grayscale image**: The original image is converted to grayscale using `cv2.cvtColor`. Grayscale conversion is useful to analyze image brightness without considering chromatic information.
- **CieLab image**: The image is then converted to the CieLab color space, which separates luminance (L) from chromatic information (A and B). The CieLab space is more sensitive to visible color variations than RGB and HSV.

#### Image Visualization

- **Original image with histogram**: The original image is displayed together with its histogram to analyze the distribution of pixel values.
- **Derived images**: Several derived images are displayed, including:
  - The grayscale image.
  - The L, A, and B channels of the image in the CieLab space, representing luminance and chromatic information, respectively.

Visualizing these images helps better understand the visual characteristics of the image and how chromatic and luminance information can be separated and analyzed.


In [11]:
img = get_random_image(train_df)
grey_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
grey_img = grey_img.astype(np.uint8, copy=False)
lab_img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
plot_single_image(img, with_hist=True)
plot_multiple_images(
    [grey_img, lab_img[:, :, 0], lab_img[:, :, 1], lab_img[:, :, 2]],
    titles=["Grayscale", "L Channel", "A Channel", "B Channel"],
    max_per_row=4,
    size=(3, 3),
)

NameError: name 'get_random_image' is not defined

### Applying Global and Local Otsu Thresholding

In this part of the code, two variants of Otsu's algorithm for image binarization are implemented: a **global** version and a **local** version.

#### Function `otsu_global`

The `otsu_global` function applies global Otsu thresholding to binarize an image. Otsu thresholding is an automatic technique to determine a separation threshold between pixels based on the image intensity distribution.

**Parameters:**
- **`img`**: Input image (must be a grayscale `uint8` image).
- **`to_segment`**: Boolean parameter indicating whether to also return the segmented image (default: `False`).

**How it works:**
- The function uses `cv2.threshold` with Otsu's method to compute a global threshold and apply it to the image.
- If `to_segment=True`, the segmented image is also returned; otherwise only the binary mask is returned.

The function returns:
- **`mask`**: Binary mask resulting from Otsu thresholding.
- **`seg`**: Segmented image (if `to_segment=True`), otherwise `None`.

#### Function `otsu_local`

The `otsu_local` function applies a local (adaptive) version of thresholding, where the threshold is computed on small windows of the image using a Gaussian-weighted mean.

**Parameters:**
- **`img`**: Input image (must be a grayscale `uint8` image).
- **`to_segment`**: Boolean parameter indicating whether to also return the segmented image (default: `False`).
- **`block_size`**: Block size used to compute the threshold (default: 35).
- **`C`**: Correction parameter for adaptive thresholding (default: 10).

**How it works:**
- The function uses `cv2.adaptiveThreshold` with Gaussian adaptive thresholding to compute a local threshold for each block of the image.
- If `to_segment=True`, the segmented image is also returned; otherwise only the binary mask is returned.

The function returns:
- **`mask`**: Binary mask resulting from local thresholding.
- **`seg`**: Segmented image (if `to_segment=True`), otherwise `None`.

#### Result Visualization

The results of the two thresholding techniques (local and global) are shown on an example image. The displayed outputs include:
- The binary mask for local thresholding.
- The segmented image after local thresholding.
- The binary mask for global thresholding.
- The segmented image after global thresholding.

These operations are useful to compare the two thresholding approaches and highlight differences between global and local methods.


In [12]:
@overload
def otsu_global(
    img: UIntArray, to_segment: Literal[True]
) -> tuple[UIntArray, UIntArray]: ...


@overload
def otsu_global(
    img: UIntArray, to_segment: Literal[False] = False
) -> tuple[UIntArray, None]: ...


def otsu_global(
    img: UIntArray,
    to_segment=False,
) -> tuple[UIntArray, UIntArray | None]:
    """
    Applies global Otsu thresholding.
    Returns:
        - mask (uint8)
        - segmented image if to_segment=True, else None
    """
    # Runtime checks (ok, not typing-related)
    if img.ndim != 2:
        raise ValueError("Input image must be single channel")
    if img.dtype != np.uint8:
        raise ValueError("Input image must be of type uint8")

    # cv2.threshold returns (retval, dst)
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # At runtime OpenCV returns uint8, but help the type checker
    mask = mask.astype(np.uint8, copy=False)

    if not to_segment:
        return mask, None

    seg = cv2.bitwise_and(img, img, mask=mask)
    seg = seg.astype(np.uint8, copy=False)

    return mask, seg


@overload
def otsu_local(
    img: UIntArray, to_segment: Literal[True], block_size: int = 35, C: int = 10
) -> tuple[UIntArray, UIntArray]: ...


@overload
def otsu_local(
    img: UIntArray,
    to_segment: Literal[False] = False,
    block_size: int = 35,
    C: int = 10,
) -> tuple[UIntArray, None]: ...


def otsu_local(
    img: UIntArray, to_segment=False, block_size: int = 35, C: int = 10
) -> tuple[UIntArray, UIntArray | None]:
    "Applies local otsu thresholding (gaussian) returning uint8 mask and if to_segment is True also the segmented image"
    if len(img.shape) == 3:
        raise ValueError("Input image must be single channel")
    if img.dtype != np.uint8:
        raise ValueError("Input image must be of type uint8")
    if block_size % 2 == 0:
        warn("Block size must be odd, incrementing by 1")
        block_size += 1

    mask = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, C
    )
    mask = mask.astype(np.uint8, copy=False)
    if not to_segment:
        return mask, None

    seg = cv2.bitwise_and(img, img, mask=mask)
    seg = seg.astype(img.dtype, copy=False)
    return mask, seg


local_mask, img_local_otsu = otsu_global(grey_img, to_segment=True)
global_mask, img_global_otsu = otsu_local(
    grey_img, to_segment=True, block_size=35, C=10
)

plot_multiple_images(
    [local_mask, img_local_otsu, global_mask, img_global_otsu],
    titles=[
        "Local Otsu Threshold",
        "Image after Local Otsu",
        "Global Threshold",
        "Image after Global Otsu",
    ],
    max_per_row=2,
)

NameError: name 'overload' is not defined

### Binary Segmentation with GrabCut

In this section of the code, the **GrabCut** algorithm is applied to segment an image into two classes: foreground and background. GrabCut is a graph-based segmentation algorithm that can be used to separate the foreground from the background in an image.

#### Function `apply_grabcut`

The `apply_grabcut` function performs image segmentation using GrabCut. The process can be initialized either with a rectangle (via cropping) or with a pre-existing mask.

**Parameters:**
- **`img`**: Input image to segment.
- **`iters`**: Number of GrabCut iterations (default: 3).
- **`cropped_pixels`**: Number of pixels to crop from the borders to reduce the influence of noisy edges (default: 10).
- **`mask`**: Optional mask to initialize the process. If not provided, GrabCut automatically initializes using a rectangle (default: `None`).

**How it works:**
1. If no mask is provided, an empty mask is created and a rectangle is used to initialize the algorithm.
2. If a mask is provided, it is used for initialization without the rectangle.
3. `cv2.grabCut` is called to perform segmentation, returning a mask representing foreground and background.
4. The mask is converted to a binary mask: foreground pixels are 255 and background pixels are 0.
5. A bitwise AND operation is performed to extract only the foreground pixels.
6. If the foreground is too small, the original image is returned.

The function returns:
- **`img_segmented`**: The segmented image containing only the foreground, or the original image if the foreground is too small.

#### Result Visualization

The original and segmented images are displayed side by side. The first shows the original image, while the second shows only the extracted foreground obtained via GrabCut.

This process is useful to automatically separate the foreground (e.g., an object) from the background in an image and is particularly useful for background removal or object isolation in complex images.


In [13]:
# GrabCut binary segmentation
def apply_grabcut(
    img: UIntArray, iters=3, cropped_pixels=10, mask: None | UIntArray = None
) -> UIntArray:
    if mask is None:
        # init with rect
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        rect = (
            cropped_pixels,
            cropped_pixels,
            img.shape[1] - 2 * cropped_pixels,
            img.shape[0] - 2 * cropped_pixels,
        )
        mode = cv2.GC_INIT_WITH_RECT
    else:
        # init with mask (do NOT use rect init)
        mask = mask.astype(np.uint8, copy=False)
        rect = None
        mode = cv2.GC_INIT_WITH_MASK

    bgModel = np.zeros((1, 65), np.float64)
    fgModel = np.zeros((1, 65), np.float64)

    mask, _, _ = cv2.grabCut(img, mask, rect, bgModel, fgModel, iters, mode)

    # Convert to binary mask (1 foreground, 0 background)
    mask2 = np.where((mask == cv2.GC_BGD) | (mask == cv2.GC_PR_BGD), 0, 255).astype(
        np.uint8
    )
    img_segmented = cv2.bitwise_and(img, img, mask=mask2)
    if img_segmented.astype(bool).sum() < img.shape[0] * img.shape[1] * 3 * 0.01:
        warn("GrabCut resulted in very small foreground, returning original image")
        return img
    return img_segmented


img_segmented = apply_grabcut(img, cropped_pixels=10)
plot_multiple_images(
    [img, img_segmented], size=(5, 5), titles=["Original Image", "Definite Foreground"]
)

NameError: name 'UIntArray' is not defined

### Bilateral Filtering -> Mean Shift Segmentation -> GrabCut

In this section of the code, a three-stage segmentation process is performed: **Bilateral Filtering**, **Mean Shift Filtering**, and **GrabCut Segmentation**. Each stage helps improve segmentation quality and separate the foreground from the background.

#### Function `seg_mean_shift_based`

The `seg_mean_shift_based` function applies a sequence of filtering and segmentation operations to isolate the foreground in an image:

1. **Bilateral filtering**: Used to reduce noise while preserving edges. The filter considers both spatial distance and color-intensity differences to determine how much a pixel should be blended with its neighbors. Parameters:
   - **`d=15`**: Filter size.
   - **`sigmaColor=50`**: Standard deviation for color differences.
   - **`sigmaSpace=75`**: Standard deviation for spatial distance.

2. **Mean Shift filtering**: Applied to the bilaterally filtered image. Mean Shift groups pixels based on spatial and chromatic similarity, creating more homogeneous regions.

3. **GrabCut segmentation**: Finally, **GrabCut** is applied to the Mean Shift result to separate foreground from background. GrabCut refines the segmentation, further improving object-background separation.

**Parameters:**
- **`img`**: Input image to segment (must be a color `uint8` image).

The function returns:
- **`bil`**: Bilaterally filtered image.
- **`img_mean_shift`**: Mean Shift filtered image.
- **`img_segmented`**: Segmented image after applying GrabCut to the Mean Shift result.

#### Result Visualization

Three images are displayed:
1. **Bilaterally filtered image**: shows the image after bilateral filtering.
2. **Mean Shift filtered image**: shows the image after Mean Shift filtering.
3. **Foreground after GrabCut**: shows the extracted foreground obtained using GrabCut on the Mean Shift filtered version.

**Note:** The process can be slow on large images, so it is recommended to use moderate-size images to avoid slowdowns.

This pipeline improves segmentation quality, especially in complex scenarios with noise or objects that are difficult to isolate from the background.


In [14]:
# Bilateral Filter -> Mean shift segmentation -> GrabCut
def seg_mean_shift_based(img: UIntArray) -> tuple[UIntArray, UIntArray, UIntArray]:
    # if color difference is > threshold not considered in smoothing, for space its the pixel distance
    bil = cv2.bilateralFilter(img, d=15, sigmaColor=50, sigmaSpace=75)
    bil = bil.astype(np.uint8, copy=False)
    img_mean_shift = cv2.pyrMeanShiftFiltering(bil, sp=50, sr=50)
    img_mean_shift = img_mean_shift.astype(np.uint8, copy=False)
    img_segmented = apply_grabcut(img_mean_shift, cropped_pixels=10)
    mask = (img_segmented.sum(axis=2) > 0).astype("uint8")
    img_segmented = cv2.bitwise_and(img, img, mask=mask)
    img_segmented = img_segmented.astype(np.uint8, copy=False)
    return bil, img_mean_shift, img_segmented


bil, img_mean_shift, img_segmented = seg_mean_shift_based(img)
plot_multiple_images(
    [bil, img_mean_shift, img_segmented],
    titles=[
        "Bilateral Filtered",
        "Mean Shift Filtered",
        "Definite Foreground after GrabCut on Mean Shift",
    ],
)

# nb: too slow on large images

NameError: name 'UIntArray' is not defined

### Bilateral Filtering -> SLIC Segmentation -> GrabCut

In this section of the code, a three-stage segmentation process is applied: **Bilateral Filtering**, **SLIC Segmentation**, and **GrabCut Segmentation**. This approach helps improve image segmentation by separating the foreground from the background more accurately.

#### Function `seg_slic_based`

The `seg_slic_based` function applies a series of segmentation operations to extract the foreground using SLIC (Simple Linear Iterative Clustering), followed by a further refinement using the GrabCut algorithm.

1. **Bilateral filtering**: The image is filtered with a bilateral filter, which reduces noise while preserving edges. Parameters:
   - **`d=15`**: Filter size.
   - **`sigmaColor=50`**: Standard deviation for color differences.
   - **`sigmaSpace=75`**: Standard deviation for spatial distance.

2. **SLIC segmentation**: The SLIC segmentation algorithm is applied to the filtered image. SLIC divides the image into homogeneous superpixels based on spatial and chromatic features. Parameters:
   - **`n_segments=100`**: Number of segments (superpixels).
   - **`compactness=10`**: Compactness parameter controlling superpixel shape.

3. **Mask creation for GrabCut**: A mask is created that identifies segments touching the image border as background, while other segments are considered as likely foreground. This mask is used as input to GrabCut.

4. **Marking segment boundaries**: Segment borders produced by SLIC are highlighted to clearly visualize superpixel separations.

5. **Applying GrabCut**: Finally, GrabCut is applied to the filtered and segmented image to refine the separation between foreground and background.

The function returns:
- **`img_bil`**: Bilaterally filtered image.
- **`img_slic_marked`**: Image with SLIC segment borders highlighted.
- **`img_grab_slic`**: Segmented image after applying GrabCut.

#### Result Visualization

Three images are displayed:
1. **Bilaterally filtered image**: shows the image after bilateral filtering.
2. **SLIC segmentation**: shows the image with SLIC superpixel borders highlighted.
3. **Foreground after GrabCut**: shows the extracted foreground obtained using GrabCut on the SLIC segmentation.

This segmentation process is useful to improve the accuracy of foreground-background separation by combining filtering and segmentation techniques.


In [15]:
# Bilateral Filter -> SLIC segmentation -> GrabCut
def seg_slic_based(
    img: UIntArray, n_segments=100, compactness=10
) -> tuple[UIntArray, UIntArray, UIntArray]:
    "Applies SLIC-based segmentation followed by GrabCut."
    # bilateral filtering
    img_bil = cv2.bilateralFilter(img, d=15, sigmaColor=50, sigmaSpace=75)
    # SLIC segmentation
    labels = slic(img_bil, n_segments=n_segments, compactness=compactness)
    # creating mask for GrabCut
    border_mask = np.zeros(labels.shape, dtype=bool)
    border_mask[0, :] = True
    border_mask[-1, :] = True
    border_mask[:, 0] = True
    border_mask[:, -1] = True
    # finding segments that touch the border
    borders_segments = np.unique(labels[border_mask])
    # creating mask: probable foreground everywhere except for border segments
    mask = np.full(img.shape[:2], fill_value=cv2.GC_PR_FGD, dtype="uint8")
    mask[np.isin(labels, borders_segments)] = cv2.GC_BGD
    # marking segment boundaries for visualization
    img_slic_marked = ski.segmentation.mark_boundaries(img_bil, labels)
    # applying GrabCut
    img_grab_slic = apply_grabcut(img, cropped_pixels=10, mask=mask)
    img_grab_slic = img_grab_slic.astype(np.uint8, copy=False)
    return img_bil, img_slic_marked, img_grab_slic  # type: ignore


img_bil, img_slic_marked, img_grab_slic = seg_slic_based(img)
plot_multiple_images(
    [img_bil, img_slic_marked, img_grab_slic],
    titles=[
        "Bilateral Filtered",
        "SLIC Segmentation",
        "Definite Foreground after GrabCut on SLIC",
    ],
)

NameError: name 'UIntArray' is not defined

### Bilateral Filtering -> SLIC Segmentation -> Normalized Cuts -> GrabCut

In this section of the code, segmentation is performed in multiple stages: **Bilateral Filtering**, **SLIC Segmentation**, **Normalized Cuts**, and **GrabCut Segmentation**. Each stage contributes to refining the separation between foreground and background.

#### Function `seg_normalized_cuts_based`

The `seg_normalized_cuts_based` function applies a multi-stage segmentation process including SLIC segmentation, the Normalized Cuts algorithm, and finally GrabCut to extract the image foreground.

1. **Bilateral filtering**: The image is first filtered with a bilateral filter to reduce noise while preserving edges. Parameters:
   - **`d=15`**: Filter size.
   - **`sigmaColor=50`**: Standard deviation for color differences.
   - **`sigmaSpace=75`**: Standard deviation for spatial distance.

2. **SLIC segmentation**: The SLIC algorithm is applied to the filtered image, splitting it into superpixels. Parameters:
   - **`n_segments=100`**: Number of segments to create.
   - **`compactness=10`**: Compactness parameter controlling superpixel shape.

3. **Normalized Cuts**: The **Normalized Cuts** algorithm is executed to further improve segmentation. Normalized Cuts seeks an optimal separation by minimizing the “cut” between pixel groups using a graph-based representation. The resulting segmentation is refined using `cut_normalized`.

4. **Mask creation for GrabCut**: A mask is created indicating which segments are likely foreground or background. Segments touching the image border are considered background.

5. **Marking segment boundaries**: The boundaries of the resulting segments are highlighted to clearly visualize superpixel borders.

6. **Applying GrabCut**: GrabCut is applied to the segmented image to extract the foreground, refining the separation between foreground and background.

The function returns:
- **`img_bil`**: Image after bilateral filtering.
- **`img_ncut_marked`**: Image with Normalized Cuts segment boundaries highlighted.
- **`img_grab_ncut`**: Segmented image after applying GrabCut.

#### Result Visualization

Three images are displayed:
1. **Bilaterally filtered image**: shows the image after filtering.
2. **Normalized Cuts segmentation**: shows the segments produced by Normalized Cuts, highlighting boundaries between superpixels.
3. **Foreground after GrabCut**: shows the extracted foreground obtained using GrabCut on the Normalized Cuts segmentation.

This approach is useful to obtain a more accurate foreground-background separation by combining segmentation and refinement techniques.


In [16]:
# Bilateral Filter -> SLIC segmentation -> Normalized Cuts -> GrabCut
def seg_normalized_cuts_based(
    img: np.ndarray, n_segments=100, compactness=10
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    "Applies SLIC-based segmentation followed by Normalized Cuts and GrabCut."
    # bilateral filtering
    img_bil = cv2.bilateralFilter(img, d=15, sigmaColor=50, sigmaSpace=75)
    # SLIC segmentation
    labels = slic(img_bil, n_segments=n_segments, compactness=compactness)
    # Normalized Cuts
    rag = rag_mean_color(img, labels, mode="similarity")
    new_labels = cut_normalized(labels, rag)
    border_mask = np.zeros(new_labels.shape, dtype=bool)
    border_mask[0, :] = True
    border_mask[-1, :] = True
    border_mask[:, 0] = True
    border_mask[:, -1] = True
    # finding segments that touch the border
    prob_bg = np.unique(new_labels[border_mask])
    # creating mask: probable foreground everywhere except for border segments
    mask = np.full(img.shape[:2], fill_value=cv2.GC_PR_FGD, dtype="uint8")
    mask[np.isin(new_labels, prob_bg)] = cv2.GC_BGD
    # marking segment boundaries for visualization
    img_ncut_marked = ski.segmentation.mark_boundaries(img_bil, new_labels)
    # applying GrabCut
    img_grab_ncut = apply_grabcut(img, cropped_pixels=10, mask=mask)
    return img_bil, img_ncut_marked, img_grab_ncut  # type: ignore


img_bil, img_ncut_marked, img_grab_ncut = seg_normalized_cuts_based(img)
plot_multiple_images(
    [img_bil, img_ncut_marked, img_grab_ncut],
    titles=[
        "Bilateral Filtered",
        "Normalized Cuts Segmentation",
        "Definite Foreground after GrabCut on Normalized Cuts",
    ],
)

NameError: name 'img' is not defined

### Test on 10 Random Images with Different Segmentation Techniques

In this part of the code, 10 random images from the dataset are tested using four different segmentation techniques. For each image, the following methods are applied:

1. **GrabCut**: Segmentation using the GrabCut algorithm to separate foreground from background.
2. **Mean Shift + GrabCut**: Mean Shift filtering is applied first, followed by GrabCut.
3. **SLIC + GrabCut**: The image is segmented using SLIC, then refined with GrabCut.
4. **Normalized Cuts + GrabCut**: Segmentation is performed via Normalized Cuts, followed by GrabCut.

#### How it works

- **`get_random_image(train_df)`**: Extracts a random image from the training dataset.
- **`apply_grabcut`**: Applies GrabCut to segment the image.
- **`seg_mean_shift_based`, `seg_slic_based`, `seg_normalized_cuts_based`**: Functions that apply Mean Shift, SLIC, and Normalized Cuts segmentation techniques, respectively.

#### Result Visualization

For each of the 10 tested images, the following outputs are displayed:
1. **Original Image**
2. **Image segmented with GrabCut**
3. **Image segmented with Mean Shift + GrabCut**
4. **Image segmented with SLIC + GrabCut**
5. **Image segmented with Normalized Cuts + GrabCut**

Each segmentation technique provides a different approach to separating foreground from background, and results are shown for visual comparison.

This process is useful to compare the effectiveness of different segmentation techniques on real images, observing how each method handles object-background separation in complex scenarios.


In [17]:
# testing on 10 different random images
# original - grabcut - mean shift - slic - normalized cuts
num = 3
for _ in range(num):
    img = get_random_image(train_df)
    img_segmented_grab = apply_grabcut(img, cropped_pixels=10, iters=7)
    _, _, img_segmented_mean = seg_mean_shift_based(img)
    _, _, img_segmented_slic = seg_slic_based(img)
    _, _, img_segmented_ncut = seg_normalized_cuts_based(img)

    plot_multiple_images(
        [
            img,
            img_segmented_grab,
            img_segmented_mean,
            img_segmented_slic,
            img_segmented_ncut,
        ],
        titles=[
            "Original Image",
            "GrabCut Segmentation",
            "Mean Shift + GrabCut",
            "SLIC + GrabCut",
            "Normalized Cuts + GrabCut",
        ],
    )

NameError: name 'get_random_image' is not defined

### Using a Dictionary to Cache Masks

In this part of the code, several functions are defined for managing binary masks, including generation, inversion, application of morphological operations, and computation of mask weights. Masks are used to segment and analyze images.

#### Function `get_border_mask`

The `get_border_mask` function generates a boolean mask highlighting the borders of an image with a specified width. It uses a dictionary (`mask_cache`) to store masks that have already been computed, avoiding repeated computation for masks of the same size.

**Parameters:**
- **`h`**: Image height.
- **`w`**: Image width.
- **`thickness`**: Border thickness to highlight (default: 1).

The function returns a boolean mask with highlighted borders.

#### Function `auto_invert_mask`

The `auto_invert_mask` function automatically inverts a binary mask based on the mean pixel value along the border. If the mask is lighter at the borders, it is inverted; otherwise it is left unchanged. This helps correct masks where background and foreground are swapped.

**Parameters:**
- **`mask`**: Input binary mask.
- **`border_thickness`**: Border thickness to consider (default: 1).

The function returns the mask (inverted if necessary).

#### Function `apply_morph`

The `apply_morph` function applies morphological **closing** followed by **opening** to a binary mask. These operations remove noise or small disconnected regions from the mask.

**Parameters:**
- **`mask`**: Binary mask to modify.
- **`kernel_size`**: Kernel size (default: 5).
- **`iterations`**: Number of iterations (default: 1).

The function returns the morphologically modified mask.

#### Function `compute_mask_weight`

The `compute_mask_weight` function computes a weight for a binary mask based on coverage and concentration. Coverage refers to the percentage of non-zero pixels in the mask, while concentration measures how uniformly mask pixels are distributed.

**How it works:**
- Computes mask coverage as the mean of active pixels.
- If coverage is too low or too high, assigns a low score.
- Computes dispersion of active pixels to measure mask concentration.

The function returns a score representing mask quality based on coverage and concentration.

#### Function `compute_weights_for_masks`

The `compute_weights_for_masks` function computes normalized weights for a list of masks. Each mask is evaluated using `compute_mask_weight`, and weights are normalized so that their sum is 1.

**Parameters:**
- **`masks`**: List of masks for which to compute weights.

The function returns an array of normalized weights for each mask.

---

These functions are useful to improve management of binary masks by computing weights and applying morphological operations to refine masks used for image segmentation.


In [18]:
# using a dict to cache masks
mask_cache: dict[tuple[int, int, int], BoolArray] = {}


def get_border_mask(h: int, w: int, thickness=1) -> BoolArray:
    "Return a boolean mask with True on the border of given thickness."
    mask = np.zeros((h, w), dtype=np.uint8)
    t = max(1, int(thickness))
    if mask_cache.get((h, w, t)) is not None:
        return mask_cache[(h, w, t)]
    mask = cv2.rectangle(mask, (0, 0), (w - 1, h - 1), 255, thickness=t)
    mask = mask.astype(bool, copy=False)
    mask_cache[(h, w, t)] = mask
    return mask


def auto_invert_mask(mask: UIntArray, border_thickness=1) -> UIntArray:
    border_mask = get_border_mask(
        mask.shape[0], mask.shape[1], thickness=border_thickness
    )
    mean_border_value = np.mean(mask[border_mask])
    # print("Mean border:", mean_border_value)
    # print("Mean border:", mean_border_value, " Overall mean:", np.mean(mask))
    if mean_border_value > 0.7 * 255:
        return ski.util.invert(mask)
    elif (mean_border_value > 0.5 * 255) and (np.mean(mask) > 0.6 * 255):
        # we arent so sure, we can check the overall average
        return ski.util.invert(mask)
    return mask


def apply_morph(mask: UIntArray, kernel_size=5, iterations=1) -> UIntArray:
    "Apply morphological closing followed by opening to the mask."
    if mask.dtype == bool:
        raise ValueError("Input mask must be of type uint8")
    elif mask.max() <= 1:
        raise ValueError("Input mask must have values in [0, 255] not [0, 1]")

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, iterations=iterations)
    opened = opened.astype(np.uint8, copy=False)
    return opened


def compute_mask_weight(mask: UIntArray) -> float:
    """Returns a weight for the given binary mask based on its coverage and concentration."""
    if mask.ndim != 2:
        raise ValueError("Mask must be 2D")
    if mask.dtype != np.uint8 and mask.dtype != bool:
        raise ValueError("Mask must be of type uint8 or bool")
    mask_bool = (mask > 0) if mask.dtype != bool else mask

    # average number of pixels covered
    coverage = mask_bool.mean()

    # low score if coverage is too low or too high
    if coverage < 0.01:
        return 0.0
    if coverage > 0.60:
        return 0.1

    coords = np.argwhere(mask_bool)
    if len(coords) < 10:
        return 0.0

    H, W = mask_bool.shape
    # Calculate dispersion as normalized std
    std_y = coords[:, 0].std() / (H / 2)
    std_x = coords[:, 1].std() / (W / 2)
    # lower values means more concentration
    dispersion = (std_y + std_x) / 2
    concentration = 1.0 - min(1.0, dispersion)

    # we search for a medium coverage with high concentration
    coverage_score = 1.0 - abs(coverage - 0.25) * 2
    coverage_score = max(0.0, coverage_score)

    weight = 0.5 * concentration + 0.5 * coverage_score
    return weight


def compute_weights_for_masks(masks: list) -> np.ndarray:
    """
    Calcola pesi normalizzati per una lista di maschere.
    """
    scores = [compute_mask_weight(m) for m in masks]
    scores = np.array(scores, dtype=np.float32)

    # If all zero, equal weights
    if scores.sum() < 0.01:
        return np.ones(len(masks)) / len(masks)

    # normalize
    weights = scores / scores.sum()
    return weights

NameError: name 'BoolArray' is not defined

### Advanced Segmentation with Otsu on LAB Channels

In this part of the code, an advanced segmentation process is applied using Otsu thresholding on each channel of the **CieLab** color model (L, A, B). Each channel is processed separately, and the resulting masks are weighted and combined to produce a final saliency mask.

#### Function `advanced_otsu_mask`

The `advanced_otsu_mask` function performs advanced segmentation using Otsu thresholding for each CieLab channel. The function follows these steps:

1. **Input image validation**: The input image must be a 3-channel **CieLab** `uint8` image.

2. **Otsu thresholding on each channel**: For each channel (L, A, B), global Otsu thresholding is applied using `otsu_global`, producing a binary mask for each channel.

3. **Mask inversion**: The obtained masks are automatically inverted (if needed) using `auto_invert_mask`. The function inverts masks if the average border value exceeds a threshold, helping correct incorrect masks.

4. **Computing mask weights**: `compute_weights_for_masks` computes weights for the three masks (L, A, B) based on coverage and concentration. These weights are used to combine channels into a single image.

5. **Combining images**: L, A, and B channels are combined and weighted according to the obtained masks. The resulting image has higher values in more salient segments.

6. **Otsu thresholding on the combined image**: Otsu thresholding is applied to the combined image to obtain a final saliency mask.

7. **Final mask inversion and morphology**: The final mask is inverted (if necessary) and morphological closing and opening are applied to refine the result.

The function returns the **final mask** highlighting the foreground, using a weighted combination of LAB channels.

#### Visualization and Results

This advanced segmentation technique yields a more precise segmentation by leveraging color information in the CieLab space and combining channel-specific masks to produce a high-quality final segmentation.


In [19]:
def advanced_otsu_mask(
    img: UIntArray,
) -> UIntArray:
    if img.ndim != 3:
        raise ValueError("Input image must be 3-channel LAB")
    if img.dtype != np.uint8:
        raise ValueError("Input image must be of type uint8")

    l_ch = img[:, :, 0]
    a_ch = img[:, :, 1]
    b_ch = img[:, :, 2]
    l_mask, _ = otsu_global(l_ch)
    a_mask, _ = otsu_global(a_ch)
    b_mask, _ = otsu_global(b_ch)

    l_mask = auto_invert_mask(l_mask, border_thickness=5)
    a_mask = auto_invert_mask(a_mask, border_thickness=5)
    b_mask = auto_invert_mask(b_mask, border_thickness=5)

    # l_mask = apply_morph(l_mask)
    # a_mask = apply_morph(a_mask)
    # b_mask = apply_morph(b_mask)

    weights = compute_weights_for_masks([l_mask, a_mask, b_mask])
    combined_img = (
        weights[0] * l_ch * l_mask / 255
        + weights[1] * a_ch * a_mask / 255
        + weights[2] * b_ch * b_mask / 255
    ).astype(np.uint8, copy=False)
    mask, _ = otsu_global(combined_img, to_segment=True)
    mask = auto_invert_mask(mask, border_thickness=10)
    mask = apply_morph(mask, kernel_size=5, iterations=2)
    return mask

NameError: name 'UIntArray' is not defined

### Advanced Segmentation with GrabCut and Superpixels

In this part of the code, advanced image segmentation is performed using a combination of **bilateral filtering**, **SLIC segmentation**, **Normalized Cuts**, and a set of analyses based on **superpixels**. The process culminates in applying **GrabCut** to refine foreground segmentation. Each segment is evaluated using a score based on several factors, including Otsu overlap, color distance from the background, centrality, and compactness.

#### Function `advanced_grabcut`

The `advanced_grabcut` function performs the following steps to obtain an advanced segmentation:

1. **Bilateral filtering**: Apply a bilateral filter to reduce noise while preserving edges.

2. **Conversion to grayscale and CieLab**: Convert the image to grayscale and to the CieLab color space.

3. **SLIC segmentation**: Segment the image using SLIC (Simple Linear Iterative Clustering), which divides the image into superpixels.

4. **Normalized Cuts**: Perform segmentation with Normalized Cuts, which aims to separate segments by optimizing the graph “cut” between pixel groups.

5. **Advanced Otsu mask**: Apply an advanced Otsu mask on the L, A, and B channels to obtain a binary separation between foreground and background.

6. **Centrality and compactness analysis**: For each segment, compute centrality (closeness to image center) and compactness (segment shape).

7. **Segment scoring**: Compute a final score for each segment based on a combination of factors, including Otsu overlap, distance from background color, centrality, and compactness.

8. **Selecting relevant segments**: Segments with high scores are selected as foreground, while low-score segments are considered background.

9. **Creating the GrabCut mask**: Create a GrabCut mask using selected foreground segments and treating the others as background.

10. **Applying GrabCut**: Perform the final segmentation using GrabCut to refine the separation between foreground and background.

#### Result Visualization

For each tested image, the following are displayed:
1. **Original Image**: The input image.
2. **Foreground after GrabCut**: The extracted foreground obtained via the advanced GrabCut procedure.
3. **Baseline GrabCut segmentation**: A basic GrabCut segmentation without advanced techniques.

#### Analysis Results

If `plot_analysis=True` is enabled, the analysis visualizations are also produced: segments with highlighted centers, the Otsu mask, and the final foreground mask.

This process is useful to improve segmentation quality, especially in images with complex objects and noise, by progressively refining the separation between foreground and background through a combination of segmentation techniques and morphological analysis.


In [20]:
def advanced_grabcut(img: UIntArray, plot_analysis=False) -> None:
    H, W = img.shape[:2]
    img_center = np.array([W / 2, H / 2])
    max_dist = np.sqrt((W / 2) ** 2 + (H / 2) ** 2)

    img_bil = cv2.bilateralFilter(img, d=15, sigmaColor=50, sigmaSpace=75)
    img_bil = img_bil.astype(np.uint8, copy=False)

    img_grey = cv2.cvtColor(img_bil, cv2.COLOR_RGB2GRAY)
    img_grey = img_grey.astype(np.uint8, copy=False)

    img_lab = cv2.cvtColor(img_bil, cv2.COLOR_RGB2LAB)
    img_lab = img_lab.astype(np.uint8, copy=False)
    superpixel_mask = slic(img_bil, n_segments=100, compactness=10)
    rag = rag_mean_color(img_bil, superpixel_mask, mode="similarity", sigma=75)
    superpixel_mask = cut_normalized(superpixel_mask, rag)

    otsu_mask = advanced_otsu_mask(img_lab)

    border_mask = get_border_mask(H, W)
    # calculating the mean color on the segment on the border
    bg_color_mean = np.zeros(3)
    border_segments = np.unique(superpixel_mask[border_mask])
    count = 0
    for seg_id in border_segments:
        mask = superpixel_mask == seg_id
        bg_color_mean += img_bil[mask].mean(axis=0)
        count += 1
    if count > 0:
        bg_color_mean /= count

    scores = []
    for seg_id in np.unique(superpixel_mask):
        mask = (superpixel_mask == seg_id).astype(np.uint8)
        moments = cv2.moments(mask)
        area = moments["m00"]
        if area < img.shape[0] * img.shape[1] * 0.005:
            # segment smaller than 0.5% of the image, skip
            continue
        # RETR_EXTERNAL to get only outer contours
        countours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not countours:
            continue
        # taking only the largest contour if SLIC produced multiple disconnected parts
        cnt = max(countours, key=cv2.contourArea)
        # true if closed
        perimeter = cv2.arcLength(cnt, True)

        # centroids calculation
        cX = int(moments["m10"] / moments["m00"])
        cY = int(moments["m01"] / moments["m00"])
        # centrality calculation
        dist = np.linalg.norm(np.array([cX, cY]) - img_center)
        centrality = 1 - (dist / max_dist)
        # compactness calculation
        if perimeter == 0:
            compactness = 0
        else:
            compactness = (4 * np.pi * area) / (perimeter**2 + 1e-6)
        # otsu overlap calculation
        otsu_overlap = np.mean(otsu_mask[superpixel_mask == seg_id])
        otsu_overlap = otsu_overlap / 255.0
        # color distance from background
        mean_col = cv2.mean(img_bil, mask=mask)[:3]
        color_dist_bg = np.linalg.norm(np.array(mean_col) - bg_color_mean)
        color_dist_bg = color_dist_bg / (np.sqrt(3 * (255**2)))

        # score calculation
        final_score = (
            (0.20 * otsu_overlap)
            + (0.35 * color_dist_bg)
            + (0.15 * centrality)
            + (0.30 * compactness)
        )

        scores.append((seg_id, final_score, cX, cY))

    # sort segments by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    if plot_analysis:
        # visualizations
        img_marked = ski.segmentation.mark_boundaries(img, superpixel_mask) * 255
        img_marked = img_marked.astype(np.uint8)
        for seg_id, score, cX, cY in scores:
            if score >= 0.6:
                cv2.circle(img_marked, (cX, cY), 5, (255, 0, 0), -1)
            elif score <= 0.2:
                cv2.circle(img_marked, (cX, cY), 5, (0, 0, 255), -1)
        # mask white to black to represent foreground
        final_mask = np.zeros(superpixel_mask.shape, dtype=np.uint8)
        for seg_id, score, _, _ in scores:
            final_mask[superpixel_mask == seg_id] = score * 255

        plot_multiple_images(
            [img_bil, img_marked, otsu_mask, final_mask],
            titles=[
                "Original Image",
                "Superpixel Segmentation with Centroids",
                "Otsu Mask",
                "Final Mask",
            ],
            max_per_row=2,
            size=(3, 3),
        )

    # now generating the mask for GrabCut
    grabcut_mask = np.full(
        superpixel_mask.shape, fill_value=cv2.GC_PR_FGD, dtype=np.uint8
    )
    # setting the border pixels as definite background, changed only if score is very high
    grabcut_mask[border_mask] = cv2.GC_BGD
    k = 5
    n = 0
    for seg_id, score, _, _ in scores:
        if n < k and score >= 0.5:
            n += 1
            # we could also check the intradifference of these segments
            # if some are too different we could mark them as probable foreground
            grabcut_mask[superpixel_mask == seg_id] = cv2.GC_FGD
        # elif score <= 0.15:
        #     grabcut_mask[superpixel_mask == seg_id] = cv2.GC_BGD
    img_segmented = apply_grabcut(img, cropped_pixels=10, mask=grabcut_mask)
    img_basic_segmented = apply_grabcut(img, cropped_pixels=10)
    # if number of pixels in segmented image is too low, we fallback to basic grabcut
    if img_segmented.astype(bool).sum() < img.shape[0] * img.shape[1] * 3 * 0.01:
        print("number of pixels in segmented image:", img_segmented.astype(bool).sum())
        print("Warning: segmented image is almost empty.")
        img_segmented = img
    plot_multiple_images(
        [img, img_segmented, img_basic_segmented],
        titles=["Original Image", "Final Definite Foreground", "Basic GrabCut"],
    )


test = 3
for _ in range(test):
    img = get_random_image(train_df_reduced)
    advanced_grabcut(img, plot_analysis=True)

NameError: name 'UIntArray' is not defined

# Generating segmented dataset


### GrabCut Segmentation and Saving Results in a DataFrame

In this section of the code, the **GrabCut** algorithm is applied to segment a set of images, with the result saved in a new DataFrame that tracks the paths of segmented images. It is also checked whether images have already been segmented, avoiding repeated processing.

#### Function `grabcut_save_from_df`

The `grabcut_save_from_df` function performs GrabCut segmentation on a set of images contained in a DataFrame and saves the results to a specified path. Images are read and processed in parallel to speed up the process, using a thread pool.

**Parameters:**
- **`df`**: The DataFrame containing the paths of images to segment.
- **`path_col`**: The name of the column in the DataFrame that contains image paths.
- **`save_path`**: The path where to save the DataFrame with segmented image paths.
- **`iters`**: The number of iterations to run GrabCut (default: 2).
- **`cropped_pixels`**: Number of pixels to crop from borders to reduce noisy border influence (default: 10).

**How it works:**
1. Image paths are read from the DataFrame.
2. For each image, **GrabCut** is applied to separate foreground from background.
3. Results are saved with a new name by adding the `_grabcut` suffix to the filename.
4. Segmented image paths are added to the DataFrame and the resulting DataFrame is saved as a CSV file.

The function returns a list of tuples, each containing the image path and a boolean indicating whether saving completed successfully.

#### Function `check_csv_done`

The `check_csv_done` function checks whether a CSV file already exists and whether all images in the DataFrame have already been processed. This prevents segmenting images more than once, saving time and resources.

**Parameters:**
- **`df`**: DataFrame containing image paths.
- **`path`**: Path to the CSV file containing segmentation results.

**How it works:**
1. Checks whether the CSV file exists and contains the `grabcut_path` column tracking segmented image paths.
2. Verifies whether all images in the DataFrame have already been segmented. If one or more images are missing, the function returns `False`.

#### Segmentation by Dataset Split

The code performs segmentation for three datasets:
1. **Training set (`train_df`)**
2. **Test set (`test_df`)**
3. **Validation set (`val_df`)**

For each split, it checks whether images are already segmented using the existing CSV file. If not, `grabcut_save_from_df` is called to perform segmentation and save results.

#### Result Visualization

Once processing is complete, the code prints the number of successful segmentations and the total for each split. If segmentations are already completed, a confirmation message is printed.

This process automates applying **GrabCut** to a large number of images, efficiently managing the data and preventing repeated processing.


In [21]:
def grabcut_save_from_df(
    df: pd.DataFrame,
    path_col: str,
    save_path: str,
    iters: int = 2,
    cropped_pixels: int = 10,
) -> list[tuple[str, bool]]:
    paths = df[path_col].tolist()
    workers = min(32, (os.cpu_count() or 4) + 4)

    print(f"Using {workers} workers for GrabCut processing.")

    def _process_one(p: str) -> tuple[str, bool]:
        try:
            in_path = Path(p)
            img = cv2.imread(str(in_path), cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("Image not found or unable to read.")
            img = img.astype(np.uint8, copy=False)

            out_rgb = apply_grabcut(
                img,
                iters=iters,
                cropped_pixels=cropped_pixels,
            )
            out_path = in_path.with_name(f"{in_path.stem}_grabcut{in_path.suffix}")
            ok = cv2.imwrite(str(out_path), out_rgb)
            return (str(out_path), ok)
        except Exception:
            return (str(p), False)

    results: list[tuple[str, bool]] = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for res in ex.map(_process_one, paths):
            results.append(res)

    df_out = df.copy(deep=True)
    df_out["grabcut_path"] = [p for p, _ in results]
    if save_path is None or save_path.strip() == "":
        save_path = "segmented_path.csv"
    df_out.to_csv(save_path, index=False)
    return results


def check_csv_done(
    df: pd.DataFrame,
    path: str,
) -> bool:
    "check if csv path exists and all the img in df are already processed"
    if not os.path.exists(path):
        return False

    df_existing = pd.read_csv(path)
    if "grabcut_path" not in df_existing.columns:
        return False
    existing_paths = set(df_existing["grabcut_path"].tolist())
    for p in df["image_path"].tolist():
        in_path = Path(p)
        out_path = in_path.with_name(f"{in_path.stem}_grabcut{in_path.suffix}")
        if str(out_path) not in existing_paths:
            print(f"Missing processed path for image: {p}")
            return False
    return True


print(len(train_df))
print(len(test_df))
print(len(val_df))

img_path_col = "image_path"

train_path = "train_segmented_paths.csv"
if not check_csv_done(train_df, train_path):
    results = grabcut_save_from_df(train_df, "image_path", train_path, iters=2)
    print("Saved OK:", sum(ok for _, ok in results), "/", len(results))
else:
    print(f"Train CSV {train_path} already processed.")

test_path = "test_segmented_paths.csv"
if not check_csv_done(test_df, test_path):
    results = grabcut_save_from_df(test_df, "image_path", test_path, iters=2)
    print("Saved OK:", sum(ok for _, ok in results), "/", len(results))
else:
    print(f"Test CSV {test_path} already processed.")

val_path = "val_segmented_paths.csv"
if not check_csv_done(val_df, val_path):
    results = grabcut_save_from_df(val_df, "image_path", val_path, iters=2)
    print("Saved OK:", sum(ok for _, ok in results), "/", len(results))
else:
    print(f"Validation CSV {val_path} already processed.")

NameError: name 'train_df' is not defined

### Visualizing Segmented Images

In this part of the code, 10 segmented images are displayed, extracted from a CSV file containing the paths of images segmented with **GrabCut**. Images are randomly selected and displayed in a single grid for comparison.

#### How it works

1. **Reading the DataFrame**: The CSV file `"train_segmented_paths.csv"` is read into a DataFrame called `train_df_seg`, which contains segmented image paths.
2. **Sampling random images**: 10 images are randomly selected from the DataFrame using the `get_random_image` function, which extracts images from the `grabcut_path` column.
3. **Displaying images**: The selected images are displayed in a grid, with 5 images per row, using `plot_multiple_images`. Each image is accompanied by a title indicating its number (e.g., "Segmented Image 1", "Segmented Image 2", etc.).

#### Result

The result is a grid of segmented images, useful for visually inspecting GrabCut segmentation results on a set of training images.

This process is useful for visual verification of segmentation quality and for comparing the extracted foreground across images.


In [ ]:
train_df_seg = pd.read_csv("train_segmented_paths.csv")
num_img = 10
img_list = []
for _ in range(num_img):
    img = get_random_image(train_df_seg, img_path_col="grabcut_path")
    img_list.append(img)

plot_multiple_images(
    img_list,
    titles=[f"Segmented Image {_ + 1}" for _ in range(num_img)],
    max_per_row=5,
)

# Dataset definitions


### Creating a Dataset to Work with PyTorch Models

In this part of the code, a `MushroomDataset` class is defined, extending PyTorch's `Dataset` class. This class is designed to load and manage a mushroom image dataset, with the option to choose between two image types (original or GrabCut-segmented). A transformation is also applied to prepare images as input to a deep learning model.

#### Class `MushroomDataset`

The `MushroomDataset` class is a subclass of `torch.utils.data.Dataset` that handles loading images and labels (edible/non-edible) for training machine learning models.

**How it works:**
1. **`__init__`**: The constructor takes as input a DataFrame containing image paths and labels, a model type (`model_type`) that determines whether to use original or GrabCut-segmented images, and an optional transformation to apply to images.

   - **`dataframe`**: DataFrame containing image paths and labels.
   - **`model_type`**: Parameter that can be 1 (original images) or 2 (GrabCut-segmented images).
   - **`transform`**: Optional transformations to apply to images (e.g., resizing, normalization).

2. **`__len__`**: Returns the length of the DataFrame, i.e., the number of images in the dataset.

3. **`__getitem__`**: Returns the image and label corresponding to the given index.
   - The image is read from the path specified in the DataFrame.
   - If the model type is 1, the original image is used; if 2, the GrabCut-segmented image is used.
   - The image is read, converted to an `Image` object, and, if a transformation is provided, the transformation is applied.
   - The image and the corresponding label (edible/non-edible) are returned.

#### Image Transformations

Two transformations are defined to prepare images as input for deep learning models:

1. **`basic_transform`**: Used for the test or validation model. It includes:
   - Resizing the image to 224x224.
   - Center cropping to 224x224 to fit the model input size without distorting aspect ratio.
   - Converting the image to a tensor.
   - Normalizing using ImageNet statistics (mean and standard deviation).

2. **`train_transform`**: Used for the training set and includes data augmentation to increase dataset variability:
   - Resizing the image to 224x224.
   - RandomResizedCrop to add variability.
   - Random horizontal flip.
   - Random rotation up to 90 degrees.
   - Converting the image to a tensor.
   - Normalizing using ImageNet statistics.

#### Usage

This class and these transformations are used to prepare the dataset and images for training a deep learning model, where images are preprocessed and normalized to be compatible with **torchvision** models.


In [ ]:
class MushroomDataset(Dataset):
    def __init__(self, dataframe, model_type: Literal[1, 2], transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.model_type = model_type

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        if self.model_type == 1:
            img_path = self.dataframe.iloc[idx]["image_path"]
        else:
            img_path = self.dataframe.iloc[idx]["grabcut_path"]
        label = self.dataframe.iloc[idx]["edible"]

        # reading and normalizing image
        image = cv2.imread(img_path, cv2.IMREAD_COLOR_RGB)
        if image is None:
            raise ValueError(f"Image not found or unable to read: {img_path}")
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)

        return image, label


# all the torchvision models expect 3-channel images of size at least 224x224
# we resize the images to 224x224 and normalize them using ImageNet statistics
image_size = 224
basic_transform = transforms.Compose(
    [
        # resize shorter side to 224
        transforms.Resize(image_size),
        # crop to 224x224 to match model input size without distorting aspect ratio
        transforms.CenterCrop(image_size),
        # convert to tensor and normalize
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_transform = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=90),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Model training and evaluating functions


### Model Training Function

The `train_model` function is designed to train a deep learning model on a dataset, monitoring loss and accuracy metrics for each epoch for both the training and validation sets. During training, it also handles saving the best model based on validation loss.

#### Function Parameters:
- **`model_type`**: Identifier that determines the model type (1 or 2).
- **`num_epochs`**: Number of training epochs.
- **`best_val_loss`**: Best validation loss achieved so far.
- **`val_loss_path`**: Path to the file where the best validation loss is saved.
- **`train_loss_list`**, **`train_acc_list`**, **`val_loss_list`**, **`val_acc_list`**: Lists storing loss and accuracy for each training and validation epoch.
- **`precision_list`**, **`recall_list`**, **`f1_list`**: Lists storing precision, recall, and F1 metrics for each epoch.
- **`confusion_matrices`**: List storing confusion matrices for each epoch.
- **`criterion`**: Loss function used for optimization.
- **`optimizer`**: Optimizer used to update model weights.
- **`scheduler`**: Learning-rate update scheduler.
- **`model`**: Model to train.
- **`train_loader`**: DataLoader for the training set.
- **`val_loader`**: DataLoader for the validation set.
- **`device`**: Device on which to run training (CPU or GPU).

#### How it works:
1. **GPU memory cleanup**: At the beginning of each epoch, GPU memory is freed to avoid out-of-memory errors.

2. **Training phase**:
   - The model is set to training mode (`model.train()`).
   - For each batch in `train_loader`, inputs and labels are loaded and moved to the device (CPU or GPU).
   - Loss is computed using the model and labels, and the optimizer updates model weights.
   - Predictions and labels are computed and stored to compute epoch-level metrics (accuracy).

3. **Training metrics computation**: At the end of each epoch, the following are computed and stored:
   - **Training loss** (`epoch_loss`)
   - **Training accuracy** (`train_acc`)

4. **Validation phase**:
   - The model is evaluated on the validation set using `evaluate_model`, which returns validation loss, accuracy, precision, recall, F1-score, and confusion matrix.

5. **Learning-rate update**: At the end of each epoch, the scheduler updates the learning rate based on validation loss.

6. **Best model saving**: If the current validation loss is better than the best so far, the model is saved as the best model and the best validation loss is updated.

#### Output:
- Training progress is printed for each epoch, showing training loss/accuracy, validation loss/accuracy, F1-score, and current learning rate.
- The best model is saved as a `.pth` file and the minimum validation loss is stored in a CSV file.

This function manages the entire training cycle, including model updates and saving the best results during training.


In [22]:
def train_model(
    model_type: Literal[1, 2],
    num_epochs: int,
    best_val_loss: float,
    val_loss_path: str,
    train_loss_list: list[float],
    train_acc_list: list[float],
    val_loss_list: list[float],
    val_acc_list: list[float],
    precision_list: list[float],
    recall_list: list[float],
    f1_list: list[float],
    confusion_matrices: list[np.ndarray],
    criterion: torch.nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
):
    # freeing GPU memory before training
    torch.cuda.empty_cache()
    for epoch in range(num_epochs):
        # training phase
        model.train()
        running_loss = 0.0
        train_preds = []
        train_labels_list = []

        loop = tqdm.tqdm(
            train_loader, leave=True, desc=f"Epoch {epoch + 1}/{num_epochs}"
        )
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)
            # BCEWithLogitsLoss expects float targets of shape (N, 1)
            labels = labels.float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            with torch.no_grad():
                preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy().flatten()
                train_preds.extend(preds)
                train_labels_list.extend(labels.cpu().numpy().flatten())

        # train metrics
        epoch_loss = running_loss / len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_loss_list.append(epoch_loss)
        train_acc_list.append(train_acc)

        # validation
        val_loss, val_acc, precision, recall, f1, cm = evaluate_model(
            model, val_loader, device, criterion
        )
        val_loss_list.append(val_loss)
        val_acc_list.append(val_acc)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        confusion_matrices.append(cm)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
            f"F1: {f1:.4f} | LR: {current_lr:.2e}"
        )

        if val_loss < best_val_loss:
            # saving the best model and best val loss to a file
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_model_{model_type}.pth")
            with open(val_loss_path, "w") as f:
                f.write(f"{best_val_loss:.4f}")
            print(f"Best model saved with validation loss: {best_val_loss:.4f}")

NameError: name 'Literal' is not defined

### Model Evaluation Function

The `evaluate_model` function is designed to evaluate a model on a validation dataset. During evaluation, several performance metrics are computed, such as loss, accuracy, precision, recall, F1-score, and the confusion matrix.

#### Function Parameters:
- **`model`**: The model to evaluate (`torch.nn.Module`).
- **`dataloader`**: DataLoader providing the validation data.
- **`device`**: Device on which to run evaluation (CPU or GPU).
- **`criterion`**: Loss function used to compute loss during evaluation.

#### How it works:
1. **Evaluation mode**: The function sets the model to evaluation mode (`model.eval()`), disabling gradient computation to reduce memory usage.

2. **Model evaluation**:
   - Evaluation is performed on each batch using the `dataloader`.
   - Images (`inputs`) and labels (`labels`) are moved to the device (CPU or GPU).
   - Loss is computed for each batch using the loss function (`criterion`).
   - Predictions are computed by applying the sigmoid function to model outputs and converting them to binary values (0 or 1) using a 0.5 threshold.

3. **Metric computation**:
   - Mean validation loss is computed as the sum of losses over all batches divided by the number of batches.
   - Performance metrics (**accuracy**, **precision**, **recall**, **F1-score**) are computed using true labels and predictions.
   - The confusion matrix is computed to visualize the number of true positives (TP), false positives (FP), true negatives (TN), and false negatives (FN).

4. **Returning results**: The function returns:
   - **`val_loss`**: Mean validation loss.
   - **`accuracy`**: Model accuracy on the validation set.
   - **`precision`**: Model precision.
   - **`recall`**: Model recall.
   - **`f1`**: Model F1-score.
   - **`cm`**: Confusion matrix.

#### Usage

This function is used to evaluate model performance on validation data after each training epoch. The returned metrics are useful to monitor training progress and identify issues such as high precision but low recall (indicating a biased model).


In [ ]:
# defining validation testing function
def evaluate_model(
    model: torch.nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    criterion: torch.nn.Module,
):
    model.eval()
    all_labels = []
    all_preds = []
    val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.astype(int).flatten())

    val_loss /= len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    # tp, fp, tn, fn
    cm = confusion_matrix(all_labels, all_preds)

    return val_loss, accuracy, precision, recall, f1, cm

### Visualizing Training and Validation Metrics

The `plot_evaluation` function is designed to plot evaluation curves during model training. The following metrics are displayed for the training and validation sets:

- **Loss** (training and validation)
- **Accuracy** (training and validation)
- **F1 Score** (validation only, useful for imbalanced problems)
- **Precision and Recall** (validation only)

#### Function Parameters:
- **`train_loss_list`**: List containing training loss for each epoch.
- **`val_loss_list`**: List containing validation loss for each epoch.
- **`train_acc_list`**: List containing training accuracy for each epoch.
- **`val_acc_list`**: List containing validation accuracy for each epoch.
- **`precision_list`**: List containing validation precision for each epoch.
- **`recall_list`**: List containing validation recall for each epoch.
- **`f1_list`**: List containing validation F1-score for each epoch.

#### How it works:
1. **X-axis (epochs)**: The x-axis represents training epochs (from first to last).

2. **Loss curve**: Training and validation loss are plotted to monitor training progress. Loss should generally decrease during training.

3. **Accuracy curve**: Training and validation accuracy are plotted to monitor how the model improves in correctly classifying examples during training and validation.

4. **F1-score curve**: F1-score is plotted only for validation, as it is important under class imbalance. F1 combines precision and recall into a single metric.

5. **Precision and recall curves**: Precision and recall are plotted for validation. These metrics are particularly useful to analyze performance under class imbalance.

6. **Visualization and saving**: A plot with all curves is generated and saved as a PNG file ("training_curves.png") at 150 DPI. The plot is also displayed.

#### Usage

This function is useful to monitor training dynamics by analyzing the balance between accuracy and loss and by tracking validation metrics such as precision, recall, and F1, which are especially important when classes are imbalanced.


In [23]:
def plot_evaluation(
    train_loss_list: list[float],
    val_loss_list: list[float],
    train_acc_list: list[float],
    val_acc_list: list[float],
    precision_list: list[float],
    recall_list: list[float],
    f1_list: list[float],
):
    epochs_range = range(1, len(train_loss_list) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Loss (Train vs Val)
    axes[0, 0].plot(epochs_range, train_loss_list, label="Train", marker="o")
    axes[0, 0].plot(epochs_range, val_loss_list, label="Validation", marker="s")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle="--", alpha=0.7)

    # Accuracy (Train vs Val)
    axes[0, 1].plot(epochs_range, train_acc_list, label="Train", marker="o")
    axes[0, 1].plot(epochs_range, val_acc_list, label="Validation", marker="s")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, linestyle="--", alpha=0.7)

    # F1 Score (Val only - most important for imbalanced)
    axes[1, 0].plot(epochs_range, f1_list, label="F1", color="purple", marker="o")
    axes[1, 0].set_title("F1 Score (Validation)")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].grid(True, linestyle="--", alpha=0.7)

    # Precision & Recall (Val)
    axes[1, 1].plot(epochs_range, precision_list, label="Precision", marker="o")
    axes[1, 1].plot(epochs_range, recall_list, label="Recall", marker="s")
    axes[1, 1].set_title("Precision & Recall (Validation)")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()

### Model Training and Evaluation Function

The `train_and_evaluate_model` function manages the complete process of training, validation, and evaluation of a deep learning model. It includes tracking best performance during training, saving and loading the best model, and final evaluation on test-set metrics.

#### Function Parameters:
- **`model_type`**: Identifier specifying which image type the model is trained on (1 for standard images, 2 for GrabCut-segmented images).
- **`num_epochs`**: Number of training epochs.
- **`criterion`**: Loss function used to optimize the model.
- **`optimizer`**: Optimizer used to update model weights.
- **`scheduler`**: Learning-rate update scheduler.
- **`device`**: Device used for training (CPU or GPU).
- **`model`**: Model to train.
- **`train_loader`**: DataLoader for the training set.
- **`val_loader`**: DataLoader for the validation set.
- **`test_loader`**: DataLoader for the test set.

#### How it works:
1. **Image type**: The function prints a message indicating whether the model will be trained on standard images or GrabCut-segmented images, based on `model_type`.

2. **Loading best validation loss**: If a file containing the previous best validation loss exists (`best_val_loss_{model_type}.txt`), it is loaded. Otherwise, an infinite validation loss is initialized to start training from scratch.

3. **Metric initialization**: Lists are created to store training and validation metrics during training:
   - Loss
   - Accuracy
   - Precision
   - Recall
   - F1-Score
   - Confusion matrix

4. **Calling the training function**: `train_model` is called to run training, while results are monitored and stored.

5. **Loading the best model**: After training, the model with the best validation loss is loaded (from `best_model_{model_type}.pth`).

6. **Final evaluation on the test set**:
   - The final evaluation is performed on the test set using `evaluate_model`.
   - Final metrics (loss, accuracy, precision, recall, F1-score) and the test confusion matrix are printed.

#### Result:
At the end, final evaluation metrics on the test set are displayed, allowing assessment of model performance in a realistic test setting.

This function is essential to manage the full training and evaluation lifecycle, enabling parameter optimization and analysis of final model performance.


In [24]:
def train_and_evaluate_model(
    model_type: Literal[1, 2],
    num_epochs: int,
    criterion: torch.nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    device: torch.device,
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
):
    if model_type == 1:
        print("Training model on classical images.")
    elif model_type == 2:
        print("Training model on grabcut segmented images.")

    if os.path.exists(f"best_val_loss_{model_type}.txt"):
        with open(f"best_val_loss_{model_type}.txt", "r") as f:
            best_val_loss = float(f.read().strip())
        print(f"Loaded best validation loss from file: {best_val_loss:.4f}")
    else:
        print("No existing best validation loss found. Starting fresh.")
        best_val_loss = float("inf")

    # defining the metrics
    train_loss_list = []
    train_acc_list = []
    val_loss_list = []
    val_acc_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    confusion_matrices = []
    train_model(
        model_type,
        num_epochs,
        best_val_loss,
        f"best_val_loss_{model_type}.txt",
        train_loss_list,
        train_acc_list,
        val_loss_list,
        val_acc_list,
        precision_list,
        recall_list,
        f1_list,
        confusion_matrices,
        criterion,
        optimizer,
        scheduler,
        model,
        train_loader,
        val_loader,
        device
    )

    model.load_state_dict(torch.load(f"best_model_{model_type}.pth"))
    test_loss, test_acc, test_prec, test_rec, test_f1, test_cm = evaluate_model(
        model, test_loader, device, criterion
    )
    print("=" * 50)
    print("FINAL TEST SET METRICS")
    print("=" * 50)
    print(f"Loss:      {test_loss:.4f}")
    print(f"Accuracy:  {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall:    {test_rec:.4f}")
    print(f"F1-Score:  {test_f1:.4f}")
    print("\nConfusion Matrix:")
    print(test_cm)
    print("=" * 50)

NameError: name 'Literal' is not defined

# Models comparison


### Creating DataLoaders for Training, Validation, and Test

In this section of the code, DataLoaders are defined for the training, validation, and test sets. They are created separately for two types of images:
1. **Standard images**
2. **GrabCut-segmented images**

#### Parameters:
- **`batch_size`**: Batch size for training and validation, set to 64.
- **`train_df`, `test_df`, `val_df`**: DataFrames containing image paths for training, test, and validation sets, respectively. These DataFrames are read from CSV files.

#### Dataset for Standard Images:
1. **`train_dataset_1`**: Training dataset for standard images.
2. **`val_dataset_1`**: Validation dataset for standard images.
3. **`test_dataset_1`**: Test dataset for standard images.

For these datasets, `train_transform` is used for training and `basic_transform` is used for validation and test.

#### Dataset for GrabCut-Segmented Images:
1. **`train_dataset_2`**: Training dataset for GrabCut-segmented images.
2. **`val_dataset_2`**: Validation dataset for GrabCut-segmented images.
3. **`test_dataset_2`**: Test dataset for GrabCut-segmented images.

For these datasets, the same `train_transform` is used for training and `basic_transform` for validation and test.

#### Creating the DataLoaders:
- **`train_loader_1`**, **`val_loader_1`**, **`test_loader_1`**: DataLoaders for standard images, created with the respective dataset and batch size. `shuffle=True` is used for the training set to shuffle data, while `shuffle=False` is used for validation and test.
- **`train_loader_2`**, **`val_loader_2`**, **`test_loader_2`**: DataLoaders for GrabCut-segmented images, created similarly.

#### Using the DataLoaders:
DataLoaders are used to load data during training, validation, and testing, ensuring efficient batch management and parallel data loading via `num_workers`.

This setup makes it easy to manage and load data efficiently during model training.


In [ ]:
# defining dataloaders
batch_size = 64
train_df = pd.read_csv("train_segmented_paths.csv")
test_df = pd.read_csv("test_segmented_paths.csv")
val_df = pd.read_csv("val_segmented_paths.csv")
# 1: classical images
train_dataset_1 = MushroomDataset(train_df, model_type=1, transform=train_transform)
val_dataset_1 = MushroomDataset(val_df, model_type=1, transform=basic_transform)
test_dataset_1 = MushroomDataset(test_df, model_type=1, transform=basic_transform)
train_loader_1 = DataLoader(train_dataset_1, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader_1 = DataLoader(val_dataset_1, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader_1 = DataLoader(test_dataset_1, batch_size=batch_size, shuffle=False, num_workers=4)
# 2: grabcut segmented images
train_dataset_2 = MushroomDataset(train_df_seg, model_type=2, transform=train_transform)
val_dataset_2 = MushroomDataset(val_df, model_type=2, transform=basic_transform)
test_dataset_2 = MushroomDataset(test_df, model_type=2, transform=basic_transform)
train_loader_2 = DataLoader(train_dataset_2, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader_2 = DataLoader(val_dataset_2, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader_2 = DataLoader(test_dataset_2, batch_size=batch_size, shuffle=False, num_workers=4)


## Full images model

### 6.2 Initialization and Adaptation of the MobileNetV3-Small Architecture

#### 6.2.1 Loading the Pre-trained Model

The **MobileNetV3-Small** architecture is loaded with weights pre-trained on ImageNet (1.28M images, 1000 classes):
```python
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)
```

**Transfer learning strategy**: Backbone (feature extractor) weights are initialized from values learned on ImageNet, allowing the use of already optimized generic visual representations (edges, textures, geometric shapes).

#### 6.2.2 Modifying the Output Layer for Binary Classification

The original architecture ends with a linear layer for 1000-class classification:
```python
# Configurazione originale
Linear(in_features=576, out_features=1000)
```

For the binary classification task (edible vs non-edible), the layer is replaced with a single neuron:
```python
model.classifier[3] = torch.nn.Linear(
    in_features=576,  # dimensione del feature vector dal backbone
    out_features=1     # singolo logit per classificazione binaria
)
```

**Rationale**: 
- Single output + `BCEWithLogitsLoss` is mathematically equivalent to two outputs + `CrossEntropyLoss`
- Higher computational efficiency (half the parameters in the final layer)
- Direct interpretation: logit > 0 → positive class (edible)

**Classifier dimensions**:
- Input size: 576 features (output of the last convolutional block)
- Output size: 1 (binary logit)

#### 6.2.3 Model Complexity Analysis
```python
total_params = sum(p.numel() for p in model.parameters())
# Output atteso: ~2,537,000 parametri
```

**Parameter distribution**:
- Backbone (feature extractor): ~2.5M parameters (can be frozen for feature extraction)
- Classifier head: ~577 parameters (always trainable)

**Comparison with other architectures**:
- MobileNetV3-Small: ~2.5M parameters
- MobileNetV3-Large: ~5.4M parameters
- ResNet-50: ~25.6M parameters
- EfficientNet-B0: ~5.3M parameters

Choosing MobileNetV3-Small balances accuracy and efficiency and is particularly suitable for:
1. Rapid experimentation (fast training)
2. Deployment on resource-constrained devices
3. Reduced overfitting risk on moderately sized datasets (~3000 training images)

#### 6.2.4 Allocating the Model to a Device
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
```

**Automatic detection**:
- If CUDA GPU is available → accelerated training (5–20× speedup vs CPU)
- Otherwise CPU fallback (guaranteed compatibility but slower training)

**Memory footprint**:
- Model parameters (FP32): ~10 MB
- Intermediate activations (batch_size=32): ~50–100 MB
- Gradients during backpropagation: ~10 MB

**Total GPU memory requirement**: ~100–150 MB (compatible even with entry-level GPUs)

#### 6.2.5 Verifying the Modified Architecture

The printed output allows verifying:
1. **Original layer**: `Linear(in_features=576, out_features=1000, bias=True)`
2. **Modified layer**: `Linear(in_features=576, out_features=1, bias=True)`
3. **Total parameters**: confirms that only the last layer was modified
4. **Device assignment**: confirms allocation to GPU/CPU

This configuration is now ready for fine-tuning on the specific mushroom classification task.


In [ ]:
# modifying the final layer, the third block, for binary classification
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)
print("Classifier layer:", model.classifier[3])
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, 1)
print("Modified Classifier layer:", model.classifier[3])
print("The model input size is:", model.classifier[0].in_features)

# checking model parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# transferring model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model transferred to device: {device}")

## Training setup

The training phase includes two alternative modes, controlled by the `FREEZE_BACKBONE` flag.  
If the flag is set to `True`, the backbone parameters (`model.features`) are frozen (`requires_grad = False`), so that only the classification layer weights (`model.classifier`) are updated. In this case, the chosen optimizer is **Adam** applied only to the classifier, with a learning rate of `1e-3`.

If instead `FREEZE_BACKBONE = False`, the entire model is optimized. To make training more stable, two parameter groups with different learning rates are used: a lower one for features extracted by the backbone (`1e-5`) and a higher one for the classifier (`2e-4`). **Adam** is also used here, configured on the two parameter groups.  
In this work, all reported experiments were run with FREEZE_BACKBONE = False, thus fine-tuning the entire model (backbone + classifier).

## Handling imbalance and loss function

Since the dataset may be imbalanced between *edible* and *non-edible* classes, a weight for the positive class (`pos_weight`) is computed from training-set frequencies. This value, if different from 1, is passed to `BCEWithLogitsLoss` to penalize errors on the under-represented class more strongly, improving the model's sensitivity to the positive class.

## Scheduler and training loop

Training is run for `num_epochs = 10` epochs. Dynamic learning-rate management is handled by `ReduceLROnPlateau`, which reduces the learning rate by a factor of `0.5` if the validation loss does not improve for `2` consecutive epochs, down to a minimum of `1e-6`.  
Finally, the complete training and evaluation procedure is started via `train_and_evaluate_model`, providing the model, dataloaders (train/validation/test), optimizer, scheduler, loss, and execution device.


In [25]:
# training setup
FREEZE_BACKBONE = False
if FREEZE_BACKBONE:
    for param in model.features.parameters():
        param.requires_grad = False
    print("Backbone frozen. Only classifier layer will be trained.")
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
else:
    print("Training entire model.")
    param_groups = [
        {"params": model.features.parameters(), "lr": 1e-5},
        {"params": model.classifier.parameters(), "lr": 2e-4},
    ]
    optimizer = optim.Adam(param_groups)

# calculating pos_weight for BCEWithLogitsLoss (value > 1)
edible_counts_train = train_df["edible"].value_counts()
pos_weight = edible_counts_train.get(False, 0) / edible_counts_train.get(True, 1)
if pos_weight != 1:
    print(f"Class weight (edible): {pos_weight:.2f}")
    # criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
else:
    criterion = torch.nn.BCEWithLogitsLoss()
# defining loss function and optimizer
num_epochs = 10
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)
criterion = torch.nn.BCEWithLogitsLoss()
train_and_evaluate_model(
    model_type=1,
    num_epochs=num_epochs,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    model=model,
    train_loader=train_loader_1,
    val_loader=val_loader_1,
    test_loader=test_loader_1,
)

Training entire model.


NameError: name 'model' is not defined

## Segmented images model
In this configuration, the same MobileNetV3-Small architecture and the same training strategy adopted for the model on original images are reused, changing only the image paths to the segmented ones obtained via GrabCut (grabcutpath column and dataloaders train_loader_2, val_loader_2, test_loader_2).
In this way, a fair comparison between the two experiments is ensured, since any performance differences can be attributed mainly to the different nature of the input (full images vs segmented images) rather than to changes in the training procedure or model architecture.


## Model adaptation for binary classification

To address a **binary classification** problem, the *MobileNetV3 Small* architecture is used with pre-trained weights (`MobileNet_V3_Small_Weights.DEFAULT`). Using a pre-trained model makes it possible to leverage already effective representations learned on large datasets, reducing convergence time and the need for labeled data.

The main intervention concerns the **classification head** (*classifier*). Specifically, the final layer (`model.classifier[3]`) is replaced with a `Linear` layer having:
- the same number of input features (`in_features` unchanged);
- **a single output unit**, consistent with the use of an implicit *sigmoid* in `BCEWithLogitsLoss` (scalar logit output).

To support the configuration, the following are printed:
- the original layer and the modified one, to verify the replacement;
- the input size of the classifier (`model.classifier[0].in_features`), useful to check compatibility with the model input.

## Parameter check and device transfer

Next, the **total number of parameters** of the model is computed by summing the elements of all tensors in `parameters()`. This indicator is useful to estimate complexity, computational cost, and overfitting risk.

Finally, the model is automatically transferred to **GPU (CUDA)** if available, otherwise to CPU. This choice maximizes performance during training and inference while keeping the code portable across different machines.


In [26]:
# modifying the final layer, the third block, for binary classification
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)
print("Classifier layer:", model.classifier[3])
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, 1)
print("Modified Classifier layer:", model.classifier[3])
print("The model input size is:", model.classifier[0].in_features)

# checking model parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# transferring model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model transferred to device: {device}")

NameError: name 'models' is not defined

## Training setup and optimization strategy

Training configuration is driven by the `FREEZE_BACKBONE` flag, which allows choosing between **full fine-tuning** and **partial training**.  

- If `FREEZE_BACKBONE = True`, all backbone parameters (`model.features`) are frozen by setting `requires_grad = False`. In this way, weight updates concern only the classifier (`model.classifier`), optimized with **Adam** and learning rate `1e-3`. This choice is typical when the dataset is small and one wants to limit overfitting while leveraging already learned features.

- If `FREEZE_BACKBONE = False`, the whole model is trained (*full fine-tuning*). To stabilize learning, two parameter groups with different learning rates are defined: a lower one for the backbone (`1e-5`) and a higher one for the classifier (`2e-4`). The idea is to partially preserve the backbone's general representations while allowing faster adaptation of the classification head.

## Handling class imbalance

To account for possible **class imbalance** in the target variable `edible`, `pos_weight` is computed from training-set frequencies. In particular, the weight is defined as the ratio between negative and positive samples, increasing the penalty for errors on the minority class.

The main correction is to use `pos_weight` directly in `BCEWithLogitsLoss` (via a tensor on `device`), rather than raising an error or ignoring imbalance. If `pos_weight = 1`, classes are balanced and the standard loss is used.

## Scheduler and training start

Training is set to `num_epochs = 10`. To make the process more robust, the `ReduceLROnPlateau` scheduler is adopted, which reduces the learning rate when the monitored metric (here the loss, with `mode="min"`) does not improve for `patience = 2` consecutive epochs, applying a reduction factor of `0.5` down to a minimum of `1e-6`.

Finally, the complete training, validation, and test procedure is executed via `train_and_evaluate_model`, using the dataloaders specific to the current configuration (`train_loader_2`, `val_loader_2`, `test_loader_2`). The correction also highlights the importance of verifying that these dataloaders are correctly defined before starting training.


In [27]:
# training setup
FREEZE_BACKBONE = False
if FREEZE_BACKBONE:
    for param in model.features.parameters():
        param.requires_grad = False
    print("Backbone frozen. Only classifier layer will be trained.")
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
else:
    print("Training entire model.")
    param_groups = [
        {"params": model.features.parameters(), "lr": 1e-5},
        {"params": model.classifier.parameters(), "lr": 2e-4},
    ]
    optimizer = optim.Adam(param_groups)

# calculating pos_weight for BCEWithLogitsLoss
edible_counts_train = train_df["edible"].value_counts()
pos_weight = edible_counts_train.get(False, 0) / edible_counts_train.get(True, 1)

# CORREZIONE: usa pos_weight nella loss function invece di sollevare un errore
if pos_weight != 1:
    print(f"Class imbalance detected. Applying class weight (edible): {pos_weight:.2f}")
    pos_weight_tensor = torch.tensor([pos_weight]).to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
else:
    print("Classes are balanced.")
    criterion = torch.nn.BCEWithLogitsLoss()

# defining optimizer and scheduler
num_epochs = 10
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)

# CORREZIONE: assicurati che i dataloader esistano
train_and_evaluate_model(
    model_type=2,
    num_epochs=num_epochs,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    model=model,
    train_loader=train_loader_2,
    val_loader=val_loader_2,
    test_loader=test_loader_2,
)

Training entire model.


NameError: name 'model' is not defined

## 7. Discussion of Results and Conclusions

In this project, a binary classification system was developed to estimate mushroom edibility from RGB images, comparing two pipelines: a model trained on the original images and a model trained on images segmented using GrabCut. The full pipeline includes automatic dataset acquisition from Kaggle, edibility label definition through bibliographic research, class reduction and balancing, and large-scale generation of a segmented dataset.

From a methodological standpoint, several segmentation techniques were explored (hierarchical SLIC, Normalized Cuts, Color Names Saliency), but GrabCut was selected as a compromise between segmentation quality, robustness, and computational cost. The same classification architecture was applied to both dataset versions (MobileNetV3-Small with ImageNet pre-trained weights), leveraging data augmentation, ImageNet-consistent normalization, and training with different learning rates for the backbone and the classifier.

The chosen metrics (accuracy, precision, recall, F1-score, and confusion matrix) allow evaluating not only overall performance, but also the model’s behavior with respect to the most critical errors from an application standpoint, particularly false positives (non-edible mushrooms classified as edible). The analysis of these metrics highlights the impact of background removal on generalization capability: the model trained on segmented images reduces its dependence on context and focuses more on the mushroom’s morphological characteristics, whereas the model trained on the original images also exploits environmental information that may be misleading. 

Overall, this work shows that integrating segmentation techniques with a lightweight, pre-trained backbone makes it possible to obtain an effective and computationally efficient classification system. Possible future developments include: more robust segmentation (e.g., deep segmentation models), extending the task to multi-class species classification, and a finer calibration of decision thresholds to further reduce the risk of false positives in real-world scenarios. 